# Extracting non-genetic data and epidemiology analysis

## Notes before running the scripts
- Before running codes, please set WORKSPACE_CDR to C2024Q3R9
- Adjust parallel threshold for polypharmacy calculation, default n_jobs=16

In [ ]:
output_dir = os.environ.get('WORKSPACE_BUCKET')

os.makedirs(f"{output_dir}/data", exist_ok=True)

os.makedirs(f"{output_dir}/results", exist_ok=True)

os.makedirs(f"{output_dir}/results/figures", exist_ok=True)

os.makedirs(f"{output_dir}/results/tables", exist_ok=True)

os.makedirs(f"{output_dir}/results/associations", exist_ok=True)

In [ ]:
# thresholds used in this study
polypharmacy_threshold = 4 # the number of medications used to determine polypharmacy

consecutive_days = 30 # the number of consecutive days used to select drug exposure and define concurrent polypharmacy
multimorbidity_threshold = 2 # the number of conditions used to determine multimorbidity
CCI_threshold = 2 # the threshold to define binary of CCI
ERV_threshold = 1 # threshold used for ERV count
IV_threshold = 1 #

# 1. Get and preprocess sociodemographics for individuals who have any diagnosis of mental and behavioural disorder (F00 - F99)


In [ ]:
F01_F99_f = f"{output_dir}/data/F01_F99_pid.csv"
#F01_F99_pid_df = get_pids_F01_F99(F01_F99_f)
F01_F99_pid_df = pd.read_csv(F01_F99_f)

In [ ]:
F01_F99_pid_df.shape

In [ ]:
print(set(F01_F99_pid_df['sex_at_birth_source_value'].to_list()))
print(set(F01_F99_pid_df['gender_source_value'].to_list()))

In [ ]:
print(set(F01_F99_pid_df['race_source_value'].to_list()))

In [ ]:
print(set(F01_F99_pid_df['gender'].to_list()))
print(set(F01_F99_pid_df['race'].to_list()))
print(set(F01_F99_pid_df['ethnicity'].to_list()))

In [ ]:
F01_F99_pids = F01_F99_pid_df['person_id'].tolist()


In [ ]:
# all the participants with demographics in the entire cohort of All of Us
out_f = f"{output_dir}/data/AoU_all_participants_demographics.csv"
#all_demographics_df = get_all_participants(out_f, save_data=True)
all_demographics_df = pd.read_csv(out_f)
print(all_demographics_df.shape)
# all_demographics_df.head()

In [ ]:
F01_F99_sociodemographic_df = all_demographics_df[all_demographics_df['person_id'].isin(F01_F99_pids)]


In [ ]:
# renaming some demographics
for name in name2dict:
    F01_F99_sociodemographic_df[name] = F01_F99_sociodemographic_df[name].replace(name2dict[name])

In [ ]:
print(F01_F99_sociodemographic_df['education_level'].value_counts())
print(F01_F99_sociodemographic_df['marital_status'].value_counts())
print(F01_F99_sociodemographic_df['income_level'].value_counts())
print(F01_F99_sociodemographic_df['sex_at_birth'].value_counts())
print(F01_F99_sociodemographic_df['race'].value_counts())
print(F01_F99_sociodemographic_df['gender'].value_counts())
print(F01_F99_sociodemographic_df['ethnicity'].value_counts())

In [ ]:
print(F01_F99_sociodemographic_df.isna().sum())
print(len(F01_F99_sociodemographic_df))
print(len(set(F01_F99_sociodemographic_df['person_id'].to_list())))

In [ ]:
# death table
output_death_f = f"{output_dir}/data/death_detail.csv"
#death_df = get_mortality_from_death(output_death_f)
death_df = pd.read_csv(output_death_f)
death_df['death_date'] = pd.to_datetime(death_df['death_date'])
print(len(death_df))
death_pids = death_df['person_id'].to_list()

In [ ]:
# duplicate found in death table
print(len(death_df))
print(len(set(death_df['person_id'].to_list())))
death_df = death_df.drop_duplicates()
len(death_df)

In [ ]:
# merge demographic df with death df
F01_F99_sociodemographic_with_death_df = pd.merge(F01_F99_sociodemographic_df, death_df[['person_id','death_date']],on='person_id', how='left')

In [ ]:
F01_F99_sociodemographic_with_death_df.isna().sum()


In [ ]:
F01_F99_sociodemographic_with_death_df['age'] = F01_F99_sociodemographic_with_death_df.apply(lambda row: calculate_age(row['year_of_birth'], row['death_date']), axis=1)


# Psychotropic medication processing


In [ ]:
all_drug_ingredent_N_output_f = f"{output_dir}/data/all_drug_ingredient_N.csv"
#all_drug_ingredent_N_df = get_drug_concept_ids(all_drug_ingredent_N_output_f)
all_drug_ingredent_N_df = pd.read_csv(all_drug_ingredent_N_output_f)
print(len(all_drug_ingredent_N_df))
print(len(set(all_drug_ingredent_N_df['concept_code'].to_list())))

In [ ]:
# drop combinations
excluded_strings = ['combinations', 'comb.', 'and psycholeptics']
drug_atc_selected = all_drug_ingredent_N_df[~all_drug_ingredent_N_df['concept_name_1'].str.contains('|'.join(excluded_strings), na=False)]

In [ ]:
print(len(drug_atc_selected))
print(len(set(drug_atc_selected['concept_code'].to_list())))

In [ ]:
all_atc_set = set(all_drug_ingredent_N_df['concept_code'].to_list())

no_comb_set = set(drug_atc_selected['concept_code'].to_list())
comb_set = all_atc_set - no_comb_set

In [ ]:
atc2name_f = f'{output_dir}/data/all_atc_N_code2name.csv'
#atc_N_code2name_df = get_all_atc_N_2name(atc2name_f)
atc_N_code2name_df = pd.read_csv(atc2name_f)
print(len(set(atc_N_code2name_df['concept_code'].to_list())))
len(atc_N_code2name_df)

In [ ]:
psychotropic_atc_lst = ['N03AF01', 'N03AG01', 'N03AX09', 'N03AX11', 'N03AX16', 'N03AX24', 'N05AA01', 'N05AA03', 'N05AB02', 
                    'N05AB03', 'N05AB04', 'N05AB06', 'N05AC02', 'N05AC03', 'N05AD01', 'N05AD08', 'N05AD10', 'N05AE02', 
                    'N05AE04', 'N05AE05', 'N05AF04', 'N05AG02', 'N05AH01', 'N05AH02', 'N05AH03', 'N05AH04', 'N05AH05', 
                    'N05AL01', 'N05AL05', 'N05AN01', 'N05AN01', 'N05AN01', 'N05AX08', 'N05AX12', 'N05AX13', 'N05AX14', 
                    'N05AX15', 'N05AX16', 'N05AX17', 'N05BA01', 'N05BA02', 'N05BA04', 'N05BA05', 'N05BA06', 'N05BA56', 
                    'N05BA09', 'N05BA12', 'N05BA13', 'N05BB01', 'N05BB51', 'N05BC01', 'N05BC51', 'N05BE01', 'N05BX05', 
                    'N05CD01', 'N05CD04', 'N05CD05', 'N05CD07', 'N05CD08', 'N05CD10', 'N05CF02', 'N05CF03', 'N05CF04', 
                    'N05CH01', 'N05CH02', 'N05CH03', 'N06AA01', 'N06AA02', 'N06AA03', 'N06AA04', 'N06AA06', 'N06AA09', 
                    'N06CA01', 'N06AA10', 'N06AA11', 'N06AA12', 'N06AA17', 'N06AA21', 'N06AB03', 'N06CA03', 'N06AB04', 
                    'N06AB05', 'N06AB06', 'N06AB08', 'N06AB10', 'N06AF01', 'N06AF03', 'N06AF04', 'N06AX01', 'N06AX02', 
                    'N06AX05', 'N06AX06', 'N06AX09', 'N06AX11', 'N06AX12', 'N06AX16', 'N06AX17', 'N06AX21', 'N06AX23', 
                    'N06AX24', 'N06AX25', 'N06AX26', 'N06BA01', 'N06BA02', 'N06BA03', 'N06BA04', 'N06BA05', 'N06BA07', 
                    'N06BA09', 'N06BA11', 'N06BA12', 'N06BA13', 'N06BA14', 'N02CX02', 'N05AA01', 'N05AA02', 
                    'N05AA03', 'N05AA04', 'N05AA05', 'N05AA06', 'N05AA07', 'N05AB01', 'N05AB02', 'N05AB03', 'N05AB04', 
                    'N05AB05', 'N05AB06', 'N05AB07', 'N05AB08', 'N05AB09', 'N05AB10', 'N05AC01', 'N05AC02', 'N05AC03', 
                    'N05AC04', 'N05AD01', 'N05AD02', 'N05AD03', 'N05AD04', 'N05AD05', 'N05AD06', 'N05AD07', 'N05AD08', 
                    'N05AD09', 'N05AD10', 'N05AE01', 'N05AE02', 'N05AE03', 'N05AE04', 'N05AE05', 'N05AF01', 'N05AF02', 
                    'N05AF03', 'N05AF04', 'N05AF05', 'N05AG01', 'N05AG02', 'N05AG03', 'N05AH01', 'N05AH02', 'N05AH03', 
                    'N05AH04', 'N05AH05', 'N05AH06', 'N05AL01', 'N05AL02', 'N05AL03', 'N05AL04', 'N05AL05', 'N05AL06', 
                    'N05AL07', 'N05AN01', 'N05AX07', 'N05AX08', 'N05AX10', 'N05AX11', 'N05AX12', 'N05AX13', 'N05AX14', 
                    'N05AX15', 'N05AX16', 'N05AX17', 'N05BA01', 'N05BA02', 'N05BA03', 'N05BA04', 'N05BA05', 'N05BA06', 
                    'N05BA07', 'N05BA08', 'N05BA09', 'N05BA10', 'N05BA11', 'N05BA12', 'N05BA13', 'N05BA14', 'N05BA15', 
                    'N05BA16', 'N05BA17', 'N05BA18', 'N05BA19', 'N05BA21', 'N05BA22', 'N05BA23', 'N05BA24', 'N05BA56', 
                    'N05BB01', 'N05BB02', 'N05BB51', 'N05BC01', 'N05BC03', 'N05BC04', 'N05BC51', 'N05BD01', 'N05BE01', 
                    'N05BX01', 'N05BX02', 'N05BX03', 'N05BX04', 'N05BX05', 'N05CD01', 'N05CD02', 'N05CD03', 'N05CD04', 
                    'N05CD05', 'N05CD06', 'N05CD07', 'N05CD08', 'N05CD09', 'N05CD10', 'N05CD11', 'N05CD12', 'N05CD13', 
                    'N05CD14', 'N05CD15', 'N05CF01', 'N05CF02', 'N05CF03', 'N05CF04', 'N05CH01', 'N05CH02', 'N05CH03', 
                    'N06AA01', 'N06AA02', 'N06AA03', 'N06AA04', 'N06AA05', 'N06AA06', 'N06AA07', 'N06AA08', 'N06AA09', 
                    'N06AA10', 'N06AA11', 'N06AA12', 'N06AA13', 'N06AA14', 'N06AA15', 'N06AA16', 'N06AA17', 'N06AA18', 
                    'N06AA19', 'N06AA21', 'N06AA23', 'N06AB02', 'N06AB03', 'N06AB04', 'N06AB05', 'N06AB06', 'N06AB07', 
                    'N06AB08', 'N06AB09', 'N06AB10', 'N06AF01', 'N06AF02', 'N06AF03', 'N06AF04', 'N06AF05', 'N06AF06', 
                    'N06AG02', 'N06AG03', 'N06AX01', 'N06AX02', 'N06AX03', 'N06AX04', 'N06AX05', 'N06AX06', 'N06AX07', 
                    'N06AX08', 'N06AX09', 'N06AX10', 'N06AX11', 'N06AX12', 'N06AX13', 'N06AX14', 'N06AX15', 'N06AX16', 
                    'N06AX17', 'N06AX18', 'N06AX19', 'N06AX21', 'N06AX22', 'N06AX23', 'N06AX24', 'N06AX25', 'N06AX26', 
                    'N06AX27', 'N06BA01', 'N06BA02', 'N06BA03', 'N06BA04', 'N06BA05', 'N06BA06', 'N06BA07', 'N06BA08', 
                    'N06BA09', 'N06BA10', 'N06BA11', 'N06BA12', 'N06BA13', 'N06BA14']

#'N01AX14',

selected_atc_lst = list(set(psychotropic_atc_lst) - comb_set)

atc4_list = [s[:5] for s in selected_atc_lst]
print(set(atc4_list))

atc4_2_name = atc_N_code2name_df[atc_N_code2name_df['concept_code'].isin(set(atc4_list))]
atc4_2_name.to_csv(f'{output_dir}/data/select_atc42name.csv', index=False)


selected_atc2name_df = atc_N_code2name_df[atc_N_code2name_df['concept_code'].isin(selected_atc_lst)]
selected_atc2name_df.to_csv(f'{output_dir}/data/select_atc52name.csv', index=False)
selected_atc2name_df = pd.read_csv(f'{output_dir}/data/select_atc52name.csv')
selected_atc5_lst = selected_atc2name_df['concept_code'].to_list()
len(selected_atc5_lst)

In [ ]:
atc_N_code2name_df


In [ ]:
drug_exposure_f = f"{output_dir}/data/drug_exposure.parquet"
#drug_exposure_df = drug_exposure_atc(selected_atc5_lst, drug_exposure_f)
drug_exposure_df = pd.read_parquet(drug_exposure_f)
print(drug_exposure_df.isna().sum())
print(len(drug_exposure_df))
drug_exposure_pids = set(drug_exposure_df['person_id'].to_list())
print(len(drug_exposure_pids))

In [ ]:
drug2atc_df = drug_exposure_df[["person_id", 'drug_concept_id',
       'drug_exposure_start_date', 'drug_exposure_end_date', 'concept_name', 'concept_class_id', 'atc_code',
       'atc_name']].drop_duplicates()

In [ ]:
# cocurrent polypharmacy
drug2atc_clean_df = drug2atc_df.dropna(subset=['drug_exposure_end_date'])
drug2atc_clean_df['drug_exposure_start_date'] = pd.to_datetime(drug2atc_clean_df['drug_exposure_start_date'])
drug2atc_clean_df['drug_exposure_end_date'] = pd.to_datetime(drug2atc_clean_df['drug_exposure_end_date'])
drug2atc_clean_df['drug_exposure_duration_days'] = (drug2atc_clean_df['drug_exposure_end_date'] - drug2atc_clean_df['drug_exposure_start_date']).dt.days
drug2atc_final_df = drug2atc_clean_df[drug2atc_clean_df['drug_exposure_duration_days'] >= consecutive_days]

print(drug2atc_final_df.isna().sum())
print(len(drug2atc_final_df))

drug_concept_ids_lst = list(set(drug2atc_final_df['atc_code'].to_list()))
print(len(drug_concept_ids_lst))
print(f'The number of drug exposure records: {len(drug2atc_final_df)}')
n_participants_drug_ATC = len(set(drug2atc_final_df['person_id'].to_list()))
print(f'Number of participants: {n_participants_drug_ATC}')

In [ ]:
# 30
drug2atc_clean_df2 = drug2atc_df.dropna(subset=['drug_exposure_end_date'])
drug2atc_clean_df2['drug_exposure_start_date'] = pd.to_datetime(drug2atc_clean_df2['drug_exposure_start_date'])
drug2atc_clean_df2['drug_exposure_end_date'] = pd.to_datetime(drug2atc_clean_df2['drug_exposure_end_date'])
drug2atc_clean_df2['drug_exposure_duration_days'] = (drug2atc_clean_df2['drug_exposure_end_date'] - drug2atc_clean_df['drug_exposure_start_date']).dt.days
drug2atc_final_df2 = drug2atc_clean_df2[drug2atc_clean_df2['drug_exposure_duration_days'] >= consecutive_days]

print(drug2atc_final_df2.isna().sum())
print(len(drug2atc_final_df2))

print(f'The number of drug exposure records: {len(drug2atc_final_df2)}')
n_participants_drug_ATC = len(set(drug2atc_final_df2['person_id'].to_list()))
print(f'Number of participants: {n_participants_drug_ATC}')

f01_f99_pids = F01_F99_sociodemographic_with_death_df['person_id'].to_list()

pids_with_drug2 = list(set(drug2atc_final_df2['person_id'].to_list()))

cohort_pid = set(f01_f99_pids) & set(pids_with_drug2)

In [ ]:
F01_F99_sociodemographic_with_death_df.columns


In [ ]:
cohort_df = F01_F99_sociodemographic_with_death_df[F01_F99_sociodemographic_with_death_df['person_id'].isin(cohort_pid)]


In [ ]:
cohort_drug_df = drug2atc_final_df2[drug2atc_final_df2['person_id'].isin(cohort_pid)]


In [ ]:
cohort_drug_df


In [ ]:
import pandas as pd
dx_year_df = cohort_drug_df.copy()

dx_year_df['drug_exposure_start_date'] = pd.to_datetime(dx_year_df['drug_exposure_start_date'])
dx_year_df['drug_exposure_end_date'] = pd.to_datetime(dx_year_df['drug_exposure_end_date'])

dx_year_df['start_year'] = dx_year_df['drug_exposure_start_date'].dt.year
dx_year_df['end_year'] = dx_year_df['drug_exposure_end_date'].dt.year

start_counts = (
    dx_year_df.groupby('start_year')['person_id']
      .nunique()
      .reset_index(name='people_with_start')
)

end_counts = (
    dx_year_df.groupby('end_year')['person_id']
      .nunique()
      .reset_index(name='people_with_end')
)

year_counts = pd.merge(start_counts, end_counts,
                       left_on='start_year', right_on='end_year',
                       how='outer')

year_counts['year'] = year_counts['start_year'].combine_first(year_counts['end_year'])
year_counts = year_counts[['year', 'people_with_start', 'people_with_end']].sort_values('year')

In [ ]:
# year_counts

In [ ]:
cohort_drug_df.to_csv(f'{output_dir}/results/tables/cohort_drug_df.csv',index=False)


In [ ]:
import pandas as pd
from collections import defaultdict
from mpire import WorkerPool


import ast
import pandas as pd

def count_concurrent_medications(df, window_size):
    """
    Count concurrent unique drug groups at each unique start date.
    
    Parameters:
        df (pd.DataFrame): Dataframe with ['drug_exposure_start_date', 'drug_exposure_end_date', 'atc_code']
        window_size (pd.Timedelta): Time window for counting concurrent use
    
    Returns:
        pd.DataFrame: Dataframe with columns ['drug_exposure_start_date', 'concurrent_count', 'merged_drug_groups']
    """
    results = []
    
    # Get unique start dates to avoid repetition
    unique_start_dates = df["drug_exposure_start_date"].unique()
    
    for start in unique_start_dates:
        # Select drugs active in the window
        
        new_df = df[(df["drug_exposure_start_date"] <= start) & (df["drug_exposure_end_date"] >= start + pd.Timedelta(days=window_size))]
        
        active_set_lst = new_df["atc_code"].to_list()

        earliest_end_date = new_df['drug_exposure_end_date'].min()

        concurrent_count = len(set(active_set_lst))
        # Store results
        results.append({
            "drug_exposure_start_date": start,
            "earliest_drug_exposure_end_date": earliest_end_date,
            "concurrent_count": concurrent_count,
            "duration (days)": (earliest_end_date - start).days + 1,
            "merged_drug_groups": set(active_set_lst)
        })

    return pd.DataFrame(results)

def safe_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)
    return val

def get_max_concurrent_medications(result_df):
    if result_df.empty:
        return {"max_count": 0, "unique_drug_groups": [], "consecutive_unique_medication_lst": []}
    
    
    result_df_sorted = result_df.sort_values('drug_exposure_start_date')

    # Convert string to set
    result_df_sorted['merged_drug_groups'] = result_df_sorted['merged_drug_groups'].apply(safe_eval)

    # Get the list of sets
    list_of_sets = result_df_sorted['merged_drug_groups'].tolist()

    unique_medication_lst = list({frozenset(s) for s in list_of_sets})
    unique_medication_lst = [set(s) for s in unique_medication_lst]    
    unique_medication_lst.sort(key=len)
    
    # Find the maximum concurrent count
    max_count = result_df["concurrent_count"].max()

    # Filter rows with max concurrent count
    max_rows = result_df[result_df["concurrent_count"] == max_count]
    
    select_row_df = max_rows[max_rows["duration (days)"] == max_rows["duration (days)"].max()]

    # Extract unique drug groups
    unique_drug_groups = []
    seen_sets = set()

    for drug_groups in select_row_df["merged_drug_groups"]:
        for drug_set in drug_groups:
            frozen_set = frozenset(drug_set)  # Convert to immutable for comparison
            if frozen_set not in seen_sets:
                seen_sets.add(frozen_set)
                unique_drug_groups.append(drug_set)

    return (max_count, unique_drug_groups, select_row_df["duration (days)"].iloc[0], unique_medication_lst)


def get_concurrent_polypharmacy(person_id, person_df, window_size):

    concurrent_polypharmacy_df = count_concurrent_medications(person_df, window_size)
    (max_polypharmacy, max_polypharmacy_atc_lst, duration, unique_medication_lst) = get_max_concurrent_medications(concurrent_polypharmacy_df)

    return (person_id, max_polypharmacy, max_polypharmacy_atc_lst, duration, unique_medication_lst)

def yearly_concurrent_polypharmacy_cohort_parallel(
    merged_df,
    n_jobs=4,
    min_days=30,
    thresholds=(2, 3, 4, 5)
):
    merged_df = merged_df.copy()
    merged_df['year'] = merged_df['drug_exposure_start_date'].dt.year

    results = []

    for year, year_df in merged_df.groupby('year'):
        person_ids = year_df['person_id'].unique()
        total_participants = len(person_ids)

        # Prepare tasks
        tasks = []
        for person_id, person_df in year_df.groupby('person_id'):
            tasks.append((person_id, person_df, min_days))

        with WorkerPool(n_jobs=n_jobs) as pool:
            outputs = pool.map(get_concurrent_polypharmacy, tasks)
                
        results_df = pd.DataFrame(
            outputs,
            columns=[
                'person_id',
                'max_polypharmacy',
                'max_polypharmacy_atc_lst',
                'duration',
                'unique_medication_lst'
            ]
        )                    
        max_value = results_df['max_polypharmacy'].max()
        print(f"Maximum polypharmacy in year {year} is {max_value}.")      
        # Build year-level summary
        for k in range(2, max_value+1, 1):
#             if k == max(thresholds):
#                 sub_df = results_df[results_df['max_polypharmacy'] >= k]
#             else:
            sub_df = results_df[results_df['max_polypharmacy'] == k]
            
            median_duration_polypharmacy = sub_df["duration"].median()
            
            count = len(sub_df)
                
            percentage = (
                count / total_participants * 100
                if total_participants > 0 else 0
            )

            results.append({
                'year': year,
#                 'n_concurrent_medications': f"{k}+" if k == max(thresholds) else k,
                'n_concurrent_medications': k,
                'median_duration': median_duration_polypharmacy,
                'participant_count': count,
                'participant_percentage': round(percentage, 2),
                'total_participants_in_year': total_participants
            })

    return pd.DataFrame(results)

In [ ]:
yearly_concurrent_polypharmacy_df = yearly_concurrent_polypharmacy_cohort_parallel(
    cohort_drug_df,
    n_jobs=16,
    min_days=30,
    thresholds=(2, 3, 4, 5))

In [ ]:
df_1994 = yearly_concurrent_polypharmacy_df[yearly_concurrent_polypharmacy_df['year']>=1994]


In [ ]:
df_1994


In [ ]:
def print_min_max(df, threshold = 4):
    print(f"Threshold: {threshold}")
    df_t = df[df['n_concurrent_medications'] == threshold]
    print(f"min: {df_t['participant_percentage'].min()}")
    print(f"max: {df_t['participant_percentage'].max()}")

In [ ]:
print_min_max(df_1994, threshold = 2)
print_min_max(df_1994, threshold = 3)
print_min_max(df_1994, threshold = 4)
print_min_max(df_1994, threshold = 5)

In [ ]:
yearly_concurrent_polypharmacy_df.to_csv(f'{output_dir}/results/tables/yearly_concurrent_polypharmacy_20260513.csv',index=False)


In [ ]:
yearly_concurrent_polypharmacy_df

In [ ]:
import pandas as pd
yearly_concurrent_polypharmacy_df = pd.read_csv(f'{output_dir}/results/tables/yearly_concurrent_polypharmacy_20260513.csv')

In [ ]:
from google.cloud import storage

bucket_name = os.environ["WORKSPACE_BUCKET"].replace("gs://", "")
client = storage.Client()

bucket = client.bucket(bucket_name)
bucket

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['mathtext.fontset'] = 'custom'      # Better control for Greek/symbols
plt.rcParams['mathtext.rm'] = 'Arial'            
plt.rcParams['mathtext.it'] = 'Arial:italic'
plt.rcParams['axes.unicode_minus'] = False

# Ensure proper sorting
poly_df = yearly_concurrent_polypharmacy_df.copy()
poly_df['year'] = poly_df['year'].astype(int)

plt.figure(figsize=(12, 6))
levels_to_plot = [2, 3, 4, 5]
poly_df = poly_df[poly_df['n_concurrent_medications'].isin(levels_to_plot)]
poly_df['n_concurrent_medications'] = poly_df['n_concurrent_medications'].astype(str)

markers = {
    '2': 'o',   # circle
    '3': 's',   # square
    '4': 'D',   # triangle up
    '5': '^'    # diamond
}

# colors = {
#     '2': '#708090',
#     '3': 'orange',
#     '4': 'green',
#     '5': 'purple'
# }
colors = {
    '2': 'orange',
    '3': 'green',
    '4': 'red',
    '5': 'purple'
}

for level, gdf in poly_df.groupby('n_concurrent_medications'):
    gdf = gdf.sort_values('year')
    plt.plot(
        gdf['year'],
        gdf['participant_percentage'],
        marker=markers.get(level, 'o'),
        color=colors.get(level, 'black'),  # avoid red here by custom colors
        label=level,
        markersize=4.5,       
        linewidth=1.5
    )

years = sorted(poly_df['year'].unique())
# plt.xticks(years, fontsize=6, rotation=90)
    
# plt.xlabel('Year', fontsize = 6)
# plt.ylabel('Participant percentage (%)', fontsize = 16)
# plt.legend(title='Concurrent medications')
# plt.tight_layout()

# plt.xlabel('Year', fontsize = 16)


# plt.ylim(top=55)
# plt.margins(x=0.02)

plt.xticks(years, fontsize=6, rotation=90)           # 5-7 pt recommended
plt.yticks(fontsize=6)

plt.xlabel('Year', fontsize=7, labelpad=8)
plt.ylabel('Participant percentage (%)', fontsize=7, labelpad=8)

plt.legend(title='Concurrent medications', 
           title_fontsize=7, 
           fontsize=6,
           loc='best')

plt.ylim(top=55)
plt.margins(x=0.02)

# os.makedirs(f"{output_dir}/results/figures", exist_ok=True)

# Set figure size to maximum allowed width (180 mm)
width_mm = 180
width_inch = width_mm / 25.4          # ≈ 7.0866 inches
height_inch = width_inch * 0.5       # 3:4 aspect ratio (adjust if needed)

plt.gcf().set_size_inches(width_inch, height_inch)
plt.tight_layout()
plt.savefig('/tmp/yearly_concurrent_polypharmacy.pdf', bbox_inches='tight')
plt.savefig('/tmp/yearly_concurrent_polypharmacy.svg', bbox_inches='tight')

for ext in ["pdf", "svg"]:
    local_path = f"/tmp/yearly_concurrent_polypharmacy.{ext}"
    plt.savefig(local_path, bbox_inches="tight")

    blob = bucket.blob(f"results/figures/yearly_concurrent_polypharmacy.{ext}")
    blob.upload_from_filename(local_path)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Ensure proper sorting
# poly_df = yearly_concurrent_polypharmacy_df[yearly_concurrent_polypharmacy_df['year']>=1995]
poly_df = yearly_concurrent_polypharmacy_df.copy()
poly_df['year'] = poly_df['year'].astype(int)


levels_to_plot = [2, 3, 4, 5]
poly_df = poly_df[poly_df['n_concurrent_medications'].isin(levels_to_plot)]

plt.figure(figsize=(12, 6))

poly_df['n_concurrent_medications'] = poly_df['n_concurrent_medications'].astype(str)

markers = {
    '2': 'o',   # circle
    '3': 's',   # square
    '4': 'D',   # triangle up
    '5': '^'    # diamond
}

colors = {
    '2': 'orange',
    '3': 'green',
    '4': 'red',
    '5': 'purple'
}

for level, gdf in poly_df.groupby('n_concurrent_medications'):
    gdf = gdf.sort_values('year')
    plt.plot(
        gdf['year'],
        gdf['median_duration'],
        marker=markers.get(level, 'o'),
        color=colors.get(level, 'black'),  # avoid red here by custom colors
        label=level,
        markersize=4,       
        linewidth=1
    )

years = sorted(poly_df['year'].unique())

# plt.xticks(years, fontsize=12, rotation=90)
# plt.xlabel('Year', fontsize = 16)
# plt.ylabel('Median duration of concurrent \n medication exposure (days)', fontsize = 16)
# plt.legend(title='Concurrent medications')
# plt.tight_layout()

# plt.xlabel('Year', fontsize = 16)


# # plt.ylim(0,200)
# plt.margins(x=0.02)

plt.xticks(years, fontsize=6, rotation=90)           # 5-7 pt recommended
plt.yticks(fontsize=6)

plt.xlabel('Year', fontsize=7, labelpad=8)
plt.ylabel('Median duration of concurrent \n medication exposure (days)', fontsize=7, labelpad=8)

plt.legend(title='Concurrent medications', 
           title_fontsize=7, 
           fontsize=6,
           loc='best')

#plt.ylim(0,200)
plt.margins(x=0.02)

# os.makedirs(f"{output_dir}/results/figures", exist_ok=True)

# Set figure size to maximum allowed width (180 mm)
width_mm = 180
width_inch = width_mm / 25.4          # ≈ 7.0866 inches
height_inch = width_inch * 0.65       # 3:4 aspect ratio (adjust if needed)

plt.gcf().set_size_inches(width_inch, height_inch)
plt.tight_layout()

plt.savefig('/tmp/yearly_concurrent_polypharmacy_duration.pdf', bbox_inches='tight')
plt.savefig('/tmp/yearly_concurrent_polypharmacy_duration.svg', bbox_inches='tight')

for ext in ["pdf", "svg"]:
    local_path = f"/tmp/yearly_concurrent_polypharmacy_duration.{ext}"
    plt.savefig(local_path, bbox_inches="tight")

    blob = bucket.blob(f"results/figures/yearly_concurrent_polypharmacy_duration.{ext}")
    blob.upload_from_filename(local_path)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Ensure proper sorting
poly_df = yearly_concurrent_polypharmacy_df[yearly_concurrent_polypharmacy_df['year']>=1994]
poly_df['year'] = poly_df['year'].astype(int)

levels_to_plot = [2, 3, 4, 5]
poly_df = poly_df[poly_df['n_concurrent_medications'].isin(levels_to_plot)]

# plt.figure(figsize=(12, 6)) #
plt.figure(figsize=(180/25.4, 90/25.4))

poly_df['n_concurrent_medications'] = poly_df['n_concurrent_medications'].astype(str)

markers = {
    '2': 'o',   # circle
    '3': 's',   # square
    '4': 'D',   # triangle up
    '5': '^'    # diamond
}

colors = {
    '2': 'orange',
    '3': 'green',
    '4': 'red',
    '5': 'purple'
}

for level, gdf in poly_df.groupby('n_concurrent_medications'):
    gdf = gdf.sort_values('year')
    plt.plot(
        gdf['year'],
        gdf['median_duration'],
        marker=markers.get(level, 'o'),
        color=colors.get(level, 'black'),  # avoid red here by custom colors
        label=level,
        markersize=4.5,       
        linewidth=1.5        
    )

years = sorted(poly_df['year'].unique())
# plt.xticks(years, fontsize=12, rotation=90)
    
# plt.xlabel('Year', fontsize = 16)
# plt.ylabel('Median duration of concurrent \n medication exposure (days)', fontsize = 16)
# plt.legend(title='Concurrent medications', loc='upper left')
# plt.tight_layout()

# plt.xlabel('Year', fontsize = 16)

# plt.ylim(0,200)
# plt.margins(x=0.02)

plt.xticks(years, fontsize=6, rotation=90)           # 5-7 pt recommended
plt.yticks(fontsize=6)

plt.xlabel('Year', fontsize=7, labelpad=8)
plt.ylabel('Median duration of concurrent \n medication exposure (days)', fontsize=7, labelpad=8)

plt.legend(title='Concurrent medications', 
           title_fontsize=7, 
           fontsize=6,
           loc='best')

plt.ylim(0,200)
plt.margins(x=0.02)

# os.makedirs(f"{output_dir}/results/figures", exist_ok=True)

# Set figure size to maximum allowed width (180 mm)
width_mm = 180
width_inch = width_mm / 25.4          # ≈ 7.0866 inches
height_inch = width_inch * 0.5       #

plt.savefig('/tmp/yearly_concurrent_polypharmacy_duration_1994.pdf', bbox_inches='tight')
plt.savefig('/tmp/yearly_concurrent_polypharmacy_duration_1994.svg', bbox_inches='tight')


for ext in ["pdf", "svg"]:
    local_path = f"/tmp/yearly_concurrent_polypharmacy_duration_1994.{ext}"
    plt.savefig(local_path, bbox_inches="tight")

    blob = bucket.blob(f"results/figures/yearly_concurrent_polypharmacy_duration_1994.{ext}")
    blob.upload_from_filename(local_path)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Copy to avoid modifying original
df = yearly_concurrent_polypharmacy_df.copy()
df['year'] = df['year'].astype(int)

# # Normalize medication levels
# def parse_level(x):
#     if isinstance(x, str) and '+' in x:
#         return int(x.replace('+', ''))
#     return int(x)

# df['med_level'] = df['n_concurrent_medications'].apply(parse_level)

# Build cumulative (>=k) percentages
cum_results = []

for year, ydf in df.groupby('year'):
    for k in [2, 3, 4, 5]:
        pct = ydf.loc[
            ydf['n_concurrent_medications'] >= k,
            'participant_percentage'
        ].sum()

        cum_results.append({
            'year': year,
            'threshold': f'≥{k}',
            'participant_percentage': pct
        })

cum_df = pd.DataFrame(cum_results)
print(cum_df)
# Plot
# plt.figure(figsize=(12, 6))
plt.figure(figsize=(180/25.4, 60/25.4))
markers = {
    '≥2': 'o',   # circle
    '≥3': 's',   # square
    '≥4': 'D',   # triangle up
    '≥5': '^'    # diamond
}

colors = {
    '≥2': 'orange',
    '≥3': 'green',
    '≥4': 'red',
    '≥5': 'purple'
}

for threshold, gdf in cum_df.groupby('threshold'):
    gdf = gdf.sort_values('year')
    plt.plot(
        gdf['year'],
        gdf['participant_percentage'],
        marker=markers.get(threshold, 'o'),
        color=colors.get(threshold, 'black'),  # avoid red here by custom colors
        label=threshold,
        markersize=4,       
        linewidth=1
    )
    
years = sorted(cum_df['year'].unique())
# plt.xticks(years, fontsize=12, rotation=90)

# plt.xlabel('Year', fontsize = 16)
# plt.ylabel('Annual prevalence (%)', fontsize = 16)

# plt.ylim(top=55)
# plt.margins(x=0.02)

# # plt.title('Yearly concurrent polypharmacy (≥30 days)', fontsize = 16)
# plt.legend(title='Concomitant\nmedications', loc="upper right")
# plt.tight_layout()


plt.xticks(years, fontsize=6, rotation=90)           # 5-7 pt recommended
plt.yticks(fontsize=6)

plt.xlabel('Year', fontsize=7, labelpad=8)
plt.ylabel('Annual prevalence (%)', fontsize=7, labelpad=8)

# plt.legend(title='Concurrent medications', 
#            title_fontsize=7, 
#            fontsize=6,
#            loc='best')
plt.legend(
    title='Concurrent medications',
    title_fontsize=7,
    fontsize=6,
    loc='upper right',          # Good position for horizontal legend
    ncol=4,                      # ← 1 row, 4 columns
    frameon=True,
    edgecolor='none',
    handlelength=1.2,
    handletextpad=0.4,
    borderpad=0.3,
    columnspacing=1.0
)

plt.ylim(top=55)
plt.margins(x=0.02)

# os.makedirs(f"{output_dir}/results/figures", exist_ok=True)

# Set figure size to maximum allowed width (180 mm)
width_mm = 180
width_inch = width_mm / 25.4          # ≈ 7.0866 inches
height_inch = width_inch * 0.5       #

plt.savefig('/tmp/yearly_concurrent_polypharmacy_cumulative.pdf', bbox_inches='tight')
plt.savefig('/tmp/yearly_concurrent_polypharmacy_cumulative.svg', bbox_inches='tight')

for ext in ["pdf", "svg"]:
    local_path = f"/tmp/yearly_concurrent_polypharmacy_cumulative.{ext}"
    plt.savefig(local_path, bbox_inches="tight")

    blob = bucket.blob(f"results/figures/yearly_concurrent_polypharmacy_cumulative.{ext}")
    blob.upload_from_filename(local_path)

plt.show()


In [ ]:
def print_prevalence_min_max(df, threshold = "≥4"):
    df = df[df['year']>=1994]
    print(f"Threshold: {threshold}")
    df_t = df[df['threshold'] == threshold]
    print(f"min: {df_t['participant_percentage'].min()}")
    print(f"max: {df_t['participant_percentage'].max()}")

In [ ]:
print_prevalence_min_max(cum_df, threshold = "≥2")
print_prevalence_min_max(cum_df, threshold = "≥3")
print_prevalence_min_max(cum_df, threshold = "≥4")
print_prevalence_min_max(cum_df, threshold = "≥5")

In [ ]:
total_counts = (
    poly_df
    .loc[poly_df['n_concurrent_medications'].isin(['2', '3', '4'])]
    .groupby('n_concurrent_medications')['participant_count']
    .sum()
    .reset_index(name='total_participants_across_all_years')
)

total_counts

In [ ]:
# cohort_drug_df

In [ ]:
import ast

def count_concurrent_medications(df, window_size):
    """
    Count concurrent unique drug groups at each unique start date.
    
    Parameters:
        df (pd.DataFrame): Dataframe with ['drug_exposure_start_date', 'drug_exposure_end_date', 'atc_code']
        window_size (pd.Timedelta): Time window for counting concurrent use
    
    Returns:
        pd.DataFrame: Dataframe with columns ['drug_exposure_start_date', 'concurrent_count', 'merged_drug_groups']
    """
    results = []
    
    # Get unique start dates to avoid repetition
    df = df.copy()
    df["drug_exposure_start_date"] = pd.to_datetime(df["drug_exposure_start_date"])
    df["drug_exposure_end_date"] = pd.to_datetime(df["drug_exposure_end_date"])

    unique_start_dates = sorted(df["drug_exposure_start_date"].unique())
    
    for start in unique_start_dates:
        
        new_df = df[(df["drug_exposure_start_date"] <= start) & (df["drug_exposure_end_date"] >= start + pd.Timedelta(days=window_size))]
        
        active_set_lst = new_df["atc_code"].to_list()

        concurrent_count = len(set(active_set_lst))
        # Store results
        results.append({
            "drug_exposure_start_date": start,
            "concurrent_count": concurrent_count,
            "merged_drug_groups": sorted(set(active_set_lst))
        })

    return pd.DataFrame(results)

def safe_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)
    return val

def get_max_concurrent_medications(result_df):
    if result_df.empty:
        return (0, [])

    result_df = result_df.copy()

    result_df["drug_exposure_start_date"] = pd.to_datetime(
        result_df["drug_exposure_start_date"]
    )

    result_df["merged_drug_groups"] = result_df["merged_drug_groups"].apply(safe_eval)

    # max concurrency
    max_count = result_df["concurrent_count"].max()

    max_rows = result_df[result_df["concurrent_count"] == max_count]
    
    unique_drug_groups = set()
                
    for groups in max_rows["merged_drug_groups"]:
        unique_drug_groups.update(groups)

    return (max_count, sorted(unique_drug_groups))

In [ ]:
def get_polypharmacy(person_df, window_size):
    total_set_lst = person_df["atc_code"].to_list()

    total_set = set(total_set_lst)
    life_time_polypharmacy = len(total_set)
    
    first_date = person_df['drug_exposure_start_date'].min() 
    last_date = person_df['drug_exposure_end_date'].max() 
    duration = (last_date - first_date).days
    
    concurrent_polypharmacy_df = count_concurrent_medications(person_df, window_size)

    (max_polypharmacy, max_polypharmacy_atc_lst) = get_max_concurrent_medications(concurrent_polypharmacy_df)

    return (life_time_polypharmacy, total_set, duration, first_date, last_date, max_polypharmacy, max_polypharmacy_atc_lst)#, unique_medication_lst)

def polypharmacy_calculation(merged_df, n_jobs, output_file, window_size=30):
    person_ids = list(set(merged_df['person_id'].to_list()))    
    def get_polypharmacy_from_df(shared_objects, person_id):
        merged_df, window_size = shared_objects
        person_df = merged_df[merged_df['person_id'] == person_id]
        
        return get_polypharmacy(person_df, window_size)  # Process the person's data 
        
    shared_objs = merged_df, window_size
    with WorkerPool(n_jobs=n_jobs, shared_objects=shared_objs) as pool:
        results = pool.map(get_polypharmacy_from_df, person_ids)
        
    columns = ['person_id', 'lifetime_polypharmacy', "total_atc_codes", "duration", 'first_drug_exposure', 'last_drug_exposure', 'concurrent_polypharmacy', 'concurrent_polypharmacy_atc_codes']#, "medication_switch_lst"]
    polypharmacy_df = pd.DataFrame(columns=columns)
     
    unit_polypharmacy_columns = set()
    
    polypharmacy_dict = {}
    for i in range(len(person_ids)):
        p_id = person_ids[i]
        (life_time_polypharmacy, total_atc, duration, first_date, last_date, max_polypharmacy, max_polypharmacy_atc) = results[i] #,medication_switchs
            
        row = [p_id, life_time_polypharmacy, total_atc, duration, first_date, last_date, max_polypharmacy, max_polypharmacy_atc]#,medication_switchs]
        # print(row)
        polypharmacy_df.loc[len(polypharmacy_df)] = row
            
    if not polypharmacy_df.empty:
        polypharmacy_df.to_csv(output_file, index=False)
    return polypharmacy_df


In [ ]:
cohort_drug_df

In [ ]:
output_polypharmacy_file = f"{output_dir}/results/ATC_psychotropic_{polypharmacy_threshold}_{consecutive_days}"
psychotropic_polypharmacy_df = polypharmacy_calculation(cohort_drug_df,n_jobs=16, output_file=output_polypharmacy_file, window_size=consecutive_days)

In [ ]:
psychotropic_polypharmacy_df['concurrent_polypharmacy_atc_codes']

In [ ]:
# target_df = cohort_polypharmacy_with_outcome_df
target_df = psychotropic_polypharmacy_df

output_f = f"{output_dir}/results/tables/Table_top_atc_concurrent_polypharmacy_{polypharmacy_threshold}_{consecutive_days}_statistics.xlsx"
target_column='concurrent_polypharmacy'
# top_n=20
top_n = None
top_atc_concurrent_polypharmcy_df = medication_distribution(atc_N_code2name_df, target_df, TOTAL_count, output_f, polypharmacy_threshold, target_column, top_n)

In [ ]:
top_atc_concurrent_polypharmcy_df

In [ ]:
cohort_polypharmacy_df = pd.merge(cohort_df, psychotropic_polypharmacy_df[['person_id','lifetime_polypharmacy', "total_atc_codes", "duration", "concurrent_polypharmacy","concurrent_polypharmacy_atc_codes", "first_drug_exposure", "last_drug_exposure"]],on='person_id', how='left')


In [ ]:
print(cohort_polypharmacy_df.isna().sum())
cohort_polypharmacy_df.head()
len(cohort_polypharmacy_df)

In [ ]:
cohort_polypharmacy_df.to_csv(f"{output_dir}/results/cohort_polypharmacy.csv")


In [ ]:
cohort_polypharmacy_df.columns

In [ ]:
cohort_polypharmacy_df = pd.read_csv(f"{output_dir}/results/cohort_polypharmacy.csv")
cohort_polypharmacy_df['total_atc_codes'] = cohort_polypharmacy_df['total_atc_codes'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else set()
)
cohort_polypharmacy_df['concurrent_polypharmacy_atc_codes'] = cohort_polypharmacy_df['concurrent_polypharmacy_atc_codes'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else set()
)

In [ ]:
cohort_pids = cohort_polypharmacy_df['person_id'].tolist()


In [ ]:
cohort_drug_exposure = drug_exposure_df[drug_exposure_df['person_id'].isin(cohort_pids)]


In [ ]:
cohort_drug_exposure

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = cohort_polypharmacy_df.copy()

# Make sure your date columns are in datetime format
df['first_drug_exposure'] = pd.to_datetime(df['first_drug_exposure'])
df['last_drug_exposure'] = pd.to_datetime(df['last_drug_exposure'])

# Distribution plots
plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
sns.histplot(df['first_drug_exposure'].dropna(), kde=False, bins=42)
plt.title('Distribution of first drug exposure', fontsize = 20)
plt.xlabel('Year', fontsize = 18)
plt.ylabel('Number of participants', fontsize = 18)

plt.subplot(1, 2, 2)
sns.histplot(df['last_drug_exposure'].dropna(), kde=False, bins=42)
plt.title('Distribution of last drug exposure', fontsize = 20)
plt.xlabel('Year', fontsize = 18)
plt.ylabel('Number of participants', fontsize = 18)

plt.tight_layout()

plt.savefig('/tmp/drug_exposure_year_distribution.pdf', bbox_inches='tight')
plt.savefig('/tmp/drug_exposure_year_distribution.svg', bbox_inches='tight')

for ext in ["pdf", "svg"]:
    local_path = f"/tmp/drug_exposure_year_distribution.{ext}"
    plt.savefig(local_path, bbox_inches="tight")

    blob = bucket.blob(f"results/figures/drug_exposure_year_distribution.{ext}")
    blob.upload_from_filename(local_path)

plt.show()

# Get earliest first_drug_exposure date
earliest_first = df['first_drug_exposure'].min()

# Get latest last_drug_exposure date
latest_last = df['last_drug_exposure'].max()

print(f"Earliest first_drug_exposure date: {earliest_first}")
print(f"Latest last_drug_exposure date: {latest_last}")

In [ ]:
# # Calculate duration in days
df['duration_days'] = (df['last_drug_exposure'] - df['first_drug_exposure']).dt.days

# Plot the distribution of durations
plt.figure(figsize=(8, 5))
sns.histplot(df['duration_days'].dropna(), bins=30, kde=True)
plt.title('Distribution of Drug Exposure Duration (days)')
plt.xlabel('Duration (days)')
plt.ylabel('Count')
plt.show()

# Optionally, some summary statistics:
print(df['duration_days'].describe())

In [ ]:
import numpy as np

df['duration_years'] = df['duration_days'] / 365

def assign_duration_group(years):
    if years <= 5:
        return '(0, 5]'
    elif 5 < years <= 10:
        return '(5, 10]'
    elif 10 < years <= 15:
        return '(10, 15]'
    elif 15 < years <= 20:
        return '(15, 20]'    
    else:
        return '(20, 40]'

# Assign groups
df['duration_group'] = df['duration_years'].apply(assign_duration_group)

# Calculate group counts and percentages
group_counts = df['duration_group'].value_counts().sort_index()
total = group_counts.sum()
group_percentages = (group_counts / total) * 100

# Combine counts and percentages into a DataFrame
duration_summary = pd.DataFrame({
    'count': group_counts,
    'percentage': group_percentages
}).sort_index()

print(duration_summary)

In [ ]:
import matplotlib.pyplot as plt

# Sort duration_summary by percentage descending
# duration_summary_sorted = duration_summary.sort_values(by='percentage', ascending=False)


# custom_order = ['(0, 5]', '(5, 10]', '(10, 15]', '(15, 20]', '(20, 25]', '(25, 30]', '(30, 35]', '(35, 40]']
custom_order = ['(0, 5]', '(5, 10]', '(10, 15]', '(15, 20]', '(20, 40]']

duration_summary.index = pd.Categorical(
    duration_summary.index,
    categories=custom_order,
    ordered=True
)

duration_summary_sorted = duration_summary.sort_index()

plt.figure(figsize=(90/25.4, 45/25.4))   # ≈ (3.5433, 1.7717) inches

bars = plt.bar(duration_summary_sorted.index, 
               duration_summary_sorted['count'], 
               color='skyblue')

# Add percentage labels on top of each bar
total = duration_summary_sorted['count'].sum()
for bar, pct in zip(bars, duration_summary_sorted['percentage']):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, 
             height + total * 0.01, 
             f'{pct:.1f}%',                    # Reduced to 1 decimal
             ha='center', va='bottom', 
             fontsize=6, rotation=0)            # 6 pt

plt.xlabel('Cohort grouped by drug exposure duration (years)', 
           fontsize=7, labelpad=6)

plt.ylabel('Number of participants', 
           fontsize=7, labelpad=6)

plt.grid(axis='y', linestyle='--', alpha=0.7, linewidth=0.5)

plt.ylim(0, duration_summary_sorted['count'].max() * 1.15)

# Tick labels - 5-6 pt
plt.xticks(fontsize=6, ha='center')   # rotated for narrow figure
plt.yticks(fontsize=6)

plt.tight_layout()



plt.savefig('/tmp/grouped_duration_distribution.pdf', bbox_inches='tight')
plt.savefig('/tmp/grouped_duration_distribution.svg', bbox_inches='tight')

for ext in ["pdf", "svg"]:
    local_path = f"/tmp/grouped_duration_distribution.{ext}"
    plt.savefig(local_path, bbox_inches="tight")

    blob = bucket.blob(f"results/figures/grouped_duration_distribution.{ext}")
    blob.upload_from_filename(local_path)

plt.tight_layout()
plt.show()

In [ ]:
df['duration_group'].value_counts()

In [ ]:
sub_df = df[df['duration_group']=='(20, 40]']
print(len(sub_df))

In [ ]:
len(sub_df[sub_df['concurrent_polypharmacy']>=5])/len(sub_df)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Copy to be safe
plot_df = df.copy()

# Total participants per duration group
denominator = (
    plot_df
    .groupby('duration_group')
    .size()
    .reset_index(name='total_participants')
)

# Count participants per duration_group & polypharmacy level
numerator = (
    plot_df
    .groupby(['duration_group', 'concurrent_polypharmacy'])
    .size()
    .reset_index(name='count')
)

# Merge and calculate percentage
percent_df = (
    numerator
    .merge(denominator, on='duration_group')
)

percent_df['percentage'] = (
    percent_df['count'] / percent_df['total_participants'] * 100
)


percent_df['duration_group'] = pd.Categorical(
    percent_df['duration_group'],
    categories=custom_order,
    ordered=True
)

In [ ]:
import matplotlib.pyplot as plt

# plt.figure(figsize=(8, 4))
plt.figure(figsize=(90/25.4, 45/25.4))

markers = {
    2: 'o',   # circle
    3: 's',   # square
    4: 'D',   # triangle up
    5: '^'    # diamond
}

colors = {
    2: 'orange',
    3: 'green',
    4: 'red',
    5: 'purple'
}


for level in [2, 3, 4, 5]:
    gdf = percent_df[
        percent_df['concurrent_polypharmacy'] == level
    ].sort_values('duration_group')

    #label = f'≥5' if level == 5 else str(level)
    
    plt.plot(
        gdf['duration_group'],
        gdf['percentage'],
        marker=markers.get(level, 'o'),
        color=colors.get(level, 'black'),  # avoid red here by custom colors
        label=level,
        markersize=3,       
        linewidth=0.5
    )
    
    # Add percentage text labels on each point
    for x, y in zip(gdf['duration_group'], gdf['percentage']):
        plt.text(x, y + 0.5, f"{y:.1f}%", ha='center', va='bottom', fontsize=5)

# plt.xlabel('Cohort grouped by medication exposure duration (years)', fontsize = 14)
# plt.ylabel('Participant percentage (%)', fontsize = 14)
# # plt.title('Concurrent polypharmacy by duration group', fontsize = 16)
# plt.legend(title='Concomitant\nmedications')
# plt.ylim(top=45)  
# plt.tight_layout()

plt.xlabel('Cohort grouped by medication exposure duration (years)', 
           fontsize=7, labelpad=6)

plt.ylabel('Participant percentage (%)', 
           fontsize=7, labelpad=6)

# plt.legend(title='Concomitant\nmedications', 
#            title_fontsize=7, 
#            fontsize=6,
#            loc='best')
plt.legend(
    title='Concomitant medications',
    title_fontsize=6,
    fontsize=5.5,
    loc='upper left',          # Good position for horizontal legend
    ncol=4,                      # ← 1 row, 4 columns
    frameon=True,
    edgecolor='none',
    handlelength=1.2,
    handletextpad=0.4,
    borderpad=0.3,
    columnspacing=1.0
)
plt.ylim(top=55)

# Tick labels (small font + rotation because figure is narrow)
plt.xticks(fontsize=6, ha='center')
plt.yticks(fontsize=6)

plt.tight_layout()

plt.savefig('/tmp/concurrent_medication_by_duration.pdf', bbox_inches='tight')
plt.savefig('/tmp/concurrent_medication_by_duration.svg', bbox_inches='tight')

for ext in ["pdf", "svg"]:
    local_path = f"/tmp/grouped_duration_distribution.{ext}"
    plt.savefig(local_path, bbox_inches="tight")

    blob = bucket.blob(f"results/figures/concurrent_medication_by_duration.{ext}")
    blob.upload_from_filename(local_path)
    
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# plt.figure(figsize=(8, 4))
plt.figure(figsize=(90/25.4, 45/25.4))

markers = {
    2: 'o',
    3: 's',
    4: 'D',
    5: '^'
}

colors = {
    2: 'orange',
    3: 'green',
    4: 'red',
    5: 'purple'
}
# Calculate cumulative prevalence (≥2, ≥3, ≥4, ≥5)
levels = [2, 3, 4, 5]

for level in levels:
    
    gdf = (
        percent_df[percent_df['concurrent_polypharmacy'] >= level]
        .groupby('duration_group', observed=True, as_index=False)['percentage']
        .sum()
        .sort_values('duration_group')
    )

    label = f'≥{level}'

    plt.plot(
        gdf['duration_group'],
        gdf['percentage'],
        marker=markers[level],
        color=colors[level],
        label=label,
        markersize=3,       
        linewidth=0.5        
    )

    # Add percentage labels
    for x, y in zip(gdf['duration_group'], gdf['percentage']):
        plt.text(x, y + 0.5, f"{y:.1f}%", ha='center', va='bottom', fontsize=5)


# plt.xlabel('Cohort grouped by medication exposure duration (years)', fontsize=14)
# plt.ylabel('Polypharmacy prevalence (%)', fontsize=14)

# plt.legend(title='Concomitant\nmedications')

plt.xlabel('Cohort grouped by medication exposure duration (years)', 
           fontsize=7, labelpad=6)

plt.ylabel('Polypharmacy prevalence (%)', 
           fontsize=7, labelpad=6)

# plt.legend(title='Concomitant\nmedications', 
#            title_fontsize=7, 
#            fontsize=6,
#            loc='best')
plt.legend(
    title='Concomitant medications',
    title_fontsize=6,
    fontsize=5.5,
    loc='upper left',          # Good position for horizontal legend
    ncol=4,                      # ← 1 row, 4 columns
    frameon=True,
    edgecolor='none',
    handlelength=1.2,
    handletextpad=0.4,
    borderpad=0.3,
    columnspacing=1.0
)
plt.ylim(top=100)

# Tick labels (small font + rotation because figure is narrow)
plt.xticks(fontsize=6, ha='center')
plt.yticks(fontsize=6)

plt.tight_layout()


plt.savefig('/tmp/concurrent_polypharmacy_by_duration.pdf', bbox_inches='tight')
plt.savefig('/tmp/concurrent_polypharmacy_by_duration.svg', bbox_inches='tight')

for ext in ["pdf", "svg"]:
    local_path = f"/tmp/concurrent_polypharmacy_by_duration.{ext}"
    plt.savefig(local_path, bbox_inches="tight")

    blob = bucket.blob(f"results/figures/concurrent_polypharmacy_by_duration.{ext}")
    blob.upload_from_filename(local_path)
    
plt.show()


In [ ]:
cohort_polypharmacy_duration_df = cohort_polypharmacy_df.merge(df, on='person_id', how='left')


In [ ]:
import numpy as np

cohort_polypharmacy_duration_df["log_exposure"] = np.log1p(cohort_polypharmacy_duration_df["duration_days"])

cohort_polypharmacy_duration_df["log_exposure_std"] = (
    cohort_polypharmacy_duration_df["log_exposure"] - cohort_polypharmacy_duration_df["log_exposure"].mean()
) / cohort_polypharmacy_duration_df["log_exposure"].std()

In [ ]:
cohort_polypharmacy_duration_df.to_csv(f"{output_dir}/results/tables/cohort_polypharmacy_duration.csv", index = False)


In [ ]:
cohort_polypharmacy_duration_df['duration_days']


In [ ]:
set(cohort_polypharmacy_df['sex_at_birth'].to_list())
cohort_polypharmacy_df['sex'] = cohort_polypharmacy_df['sex_at_birth'].replace(sex_mapping)
set(cohort_polypharmacy_df['sex'].to_list())

In [ ]:
cohort_polypharmacy_df['sex_at_birth'].replace('PMI: Skip', 'Other', inplace=True)


In [ ]:
cohort_polypharmacy_df['sex_at_birth'].value_counts()


In [ ]:
cohort_polypharmacy_df

In [ ]:
cohort_polypharmacy_df['age_group'] = cohort_polypharmacy_df['age'].apply(age_group)
set(cohort_polypharmacy_df['age_group'].to_list())

In [ ]:
cohort_polypharmacy_df['race_group'] = cohort_polypharmacy_df['race'].replace(race_mapping)


In [ ]:
set(cohort_polypharmacy_df['race_group'].tolist() )


In [ ]:
set(cohort_polypharmacy_df['education_level'].tolist())


In [ ]:
### Table 1 Sociodemographic characteristics and prevalence of polypharmacy of All of program participants

TOTAL_count = len(cohort_polypharmacy_df)


column_names = ['Characteristics', f'No.(%)', 'Lifetime polypharmacy, No.', 
                'Lifetime polypharmacy, % [95% CI]', 'Concurrent polypharmacy, No.', 
                'Concurrent polypharmacy, % [95% CI]']

sociodemographic_df = pd.DataFrame(columns = column_names)


sex_lst = ['Male', 'Female', 'Other']# 'Intersex','No matching concept', 'None Of These', 'Unknown']
age_lst = ['18-29', '30-39', '40-49', '50-59', '60-69', '>=70']
# race_lst = ['White', 'Black or African American', 'Asian', 
#                  'Another single population', 
#                  'More than one population', 'None of these', 'Unknown']
race_lst = ['Black or African American','White','Another single population',
    'More than one population',
    'Other']
#education_lst = ['CollegeAndAbove', 'CollegeOnetoThree','TwelveOrGED', 'NoHighSchoolDegree', 'Unknown']
education_lst = ["Advanced degree",
                        "College graduate", 
                        "College one to three", 
                        "Twelve or GED", 
                        "Nine through eleven", 
                        "One to eight or Never attended", 
                        "Other"]
marital_lst = ['Married', "Living with partner", 'Separated', 'Divorced', 'Widowed', 'Never married', 'Other']
#health_insurance_lst = ['Yes', 'No', 'Other']
income_lst = ['>200k','150k-200k','100k-150k','75k-100k','50k-75k','35k-50k', '25k-35k','10k-25k','<10k','Other']
# home_status_lst = ['Own', 'Rent', 'OtherArrangement', 'Other']
# country_origin_lst = ['USA', 'Other']
# house_concern_lst = ['Yes', 'No', 'Other']
# living_years_lst = ['<1', '1-2', '3-5', '6-10', '11-20', '>20', 'Other']


#row_names = sex_lst + age_lst + race_lst + education_lst + marital_lst + health_insurance_lst + income_lst + home_status_lst + country_origin_lst + house_concern_lst + living_years_lst
row_names = sex_lst + age_lst + race_lst + education_lst + marital_lst + income_lst 
groups = ["sex_at_birth", "age_group", "race_group", 'education_level', 'marital_status', 'income_level']# 'health_insurance','current_home_status', 'country_origin', 'house_concern', 'living_years']
groups2description = {"sex_at_birth": "Sex at birth", 
                      "age_group": "Age group", 
                      "race_group": "Race", 
                      'education_level': "Education level", 
                      'marital_status': "Marital status", 
                      'income_level': "Income level (household annual income)"} 
#                       'health_insurance': "Health insurance",
#                       'current_home_status': "Current home own or rent", 
#                       'country_origin': "Country origin", 
#                       'house_concern': "House concern", 
#                       'living_years': "Current Place Living Years"}
group2category_lst = {}
group2category_lst["sex_at_birth"] = sex_lst
group2category_lst["age_group"] = age_lst
group2category_lst["race_group"] = race_lst
group2category_lst["education_level"] = education_lst
group2category_lst["marital_status"] = marital_lst
# group2category_lst["health_insurance"] = health_insurance_lst
group2category_lst["income_level"] = income_lst
# group2category_lst["current_home_status"] = home_status_lst
# group2category_lst["country_origin"] = country_origin_lst
# group2category_lst["house_concern"] = house_concern_lst
# group2category_lst["living_years"] = living_years_lst


cumulative_polypharmacy2df = cohort_polypharmacy_df[cohort_polypharmacy_df['lifetime_polypharmacy'] >=polypharmacy_threshold]
t1 = len(cumulative_polypharmacy2df)
print(t1, TOTAL_count)
p1, l1, u1 = calculate_percentage_ci(t1, TOTAL_count)

concurrent_polypharmacy2df = cohort_polypharmacy_df[cohort_polypharmacy_df['concurrent_polypharmacy'] >=polypharmacy_threshold]
t2 = len(concurrent_polypharmacy2df)
p2, l2, u2 = calculate_percentage_ci(t2, TOTAL_count)

sociodemographic_df.loc[len(sociodemographic_df)] = ["Overall prevalence", f"{TOTAL_count} (100)", t1, f"{p1} [{l1}, {u1}]", t2,f"{p2} [{l2}, {u2}]"]

for group in groups:
    #print(group)
    cohort_value_counts = cohort_polypharmacy_df[group].value_counts()
    print(cohort_value_counts)
    cur_total = sum(cohort_value_counts.values)
    # print(cur_total)
    cur_lst = group2category_lst[group]
    cur_sum = 0
    sociodemographic_df.loc[len(sociodemographic_df)] = [groups2description[group], '', '', '', '', '']
    
    cumulative_val2count = cumulative_polypharmacy2df[group].value_counts()
    concurrent_val2count = concurrent_polypharmacy2df[group].value_counts()
    
    print(concurrent_val2count)
    for item in cur_lst:
        if item in cohort_value_counts:
            cur_val = cohort_value_counts[item]
            print(item, cur_val)
            cur_sum += cur_val
            percentage_str = "{:.1f}".format(cur_val/TOTAL_count * 100)
            
            if item in cumulative_val2count:
                cum_val = cumulative_val2count[item]
            else:
                cum_val = 0
            pval1,lower1, upper1 = calculate_percentage_ci(cum_val, cur_val)
            
            if item in concurrent_val2count:
                con_val = concurrent_val2count[item]
            else:
                con_val = 0
                
            pval2,lower2, upper2  = calculate_percentage_ci(con_val, cur_val)
        
            row = [item, f"{cur_val} ({percentage_str})", cum_val,  f"{pval1} [{lower1}, {upper1}]", con_val, f"{pval2} [{lower2}, {upper2}]"]
        else:
            row = [item, 0, 0, 0, 0, 0]
#         print(row)
        sociodemographic_df.loc[len(sociodemographic_df)] = row
        
    if cur_sum == cur_total and cur_sum == TOTAL_count:
        print("-------------------------------------------------")
        print(cur_total)
        print("-------------------------------------------------")
    else:
        print(f"Sum: {cur_sum}")
        raise ValueError("ERROR: Summation is not eqaul to the total value!")
print(sociodemographic_df)
sociodemographic_df.to_excel(f"{output_dir}/results/tables/Table_1_prevalence_of_polypharmacy_cohort_EHR.xlsx", index=False)

In [ ]:
duration_group_set = set(cohort_polypharmacy_duration_df['duration_group'].to_list())
duration_group_set

In [ ]:
concurrent_polypharmacy_df = cohort_polypharmacy_df[cohort_polypharmacy_df['concurrent_polypharmacy'] >= polypharmacy_threshold]


In [ ]:
F01_F99_code2name_f = f'{output_dir}/data/all_ICD_F01_F99_code2name.csv'
F01_F99_code2name_df = get_mental_disorder_icd(F01_F99_code2name_f)
F01_F99_code2name_df = pd.read_csv(F01_F99_code2name_f)
F01_F99_code2name_df.head()

In [ ]:
len(F01_F99_code2name_df)


In [ ]:
from collections import defaultdict
icd2ids = defaultdict(list)

F_icd_ids = F01_F99_code2name_df['concept_id'].tolist()
icdcode2name = {}
for c_id in F_icd_ids:
    concept_code = F01_F99_code2name_df[F01_F99_code2name_df['concept_id'] == c_id]['concept_code'].values[0]
    concept_name = F01_F99_code2name_df[F01_F99_code2name_df['concept_id'] == c_id]['concept_name'].values[0]
    if '.' in concept_code:
        icd = concept_code.split('.')[0]
        #print(concept_code)
    else:
        icd = concept_code    
        icdcode2name[icd] = concept_name
        print(concept_code, concept_name)
    icd2ids[icd].append(c_id)

In [ ]:
def determine_sample_size(z, p, d):
    n = (z*z * p *(1-p)) / (d*d)
    return n

In [ ]:
##################################################################
# preprocessing for Table 2 Prevalence of psychotropic polypharmacy by mental disorder diagnosis
cohort_pid_set = set(cohort_polypharmacy_df['person_id'].tolist())
print(len(cohort_pid_set))

from collections import defaultdict
icd_code2concept_ids = defaultdict(list)

F_icd_concept_ids = F01_F99_code2name_df['concept_id'].tolist()

icd_code2concept_name = {}
for concept_id in F_icd_concept_ids:
    concept_code = F01_F99_code2name_df[F01_F99_code2name_df['concept_id'] == concept_id]['concept_code'].values[0]
    concept_name = F01_F99_code2name_df[F01_F99_code2name_df['concept_id']==concept_id]['concept_name'].values[0]
    if '.' in concept_code:
        icd = concept_code.split('.')[0]
    else:
        icd = concept_code 
        icd_code2concept_name[icd] = concept_name
        
    icd_code2concept_ids[icd].append(concept_id)

F_icd_code2pids = {}

F_icd_code2id = {}

F_all_pids = []  
cohort_icd_lst= []
for icd_3, cid_lst in icd_code2concept_ids.items():    
    output_f = f"{output_dir}/data/icd10cm_{icd_3}_person_ids.csv"
    F_icd_person_df = extract_person_ids_omops(cid_lst, output_f, save_data=True)
    F_icd_person_df = pd.read_csv(output_f)    
    
    cur_pids = set(F_icd_person_df['person_id'].tolist())

    cur_pid_lst = list(cohort_pid_set & cur_pids)
    
    if len(cur_pid_lst) > 0:
        F_icd_code2pids[icd_3] = cur_pid_lst
        
        F_icd_code2id[icd_3] = cid_lst
        
        cohort_icd_lst.append(icd_3)
        F_all_pids.extend(cur_pid_lst)    
print(cohort_icd_lst)    
###############################
# add diagnoses to cohort_polypharmacy_df

# Step 1: Build a reverse lookup: person_id -> list of ICD codes
pid2icds = defaultdict(set)
for icd, pid_lst in F_icd_code2pids.items():
    for pid in pid_lst:
        pid2icds[pid].add(icd)

# Step 2
cohort_polypharmacy_df['psychiatric_diagnoses'] = cohort_polypharmacy_df['person_id'].apply(lambda pid: pid2icds.get(pid, []))
cohort_polypharmacy_df['psychiatric_diagnoses_count'] = cohort_polypharmacy_df['psychiatric_diagnoses'].apply(len)        
##############################

F_icd_code2pids["F01_F99"] = list(set(F_all_pids))
icd_code2concept_name["F01_F99"] = "Mental, Behavioral and Neurodevelopmental disorders"   

# print(F_icd_code2pids.keys())

########################################################
## Table 2
table2_column_names = ["ICD-10-CM diagnosis", 
                     "N", 
                     "Lifetime psychotropic polypharmacy, N",
                     "Lifetime Psychotropic polypharmacy, % [95%CI]",
                     "Concurrent psychotropic polypharmacy, N",
                     "Concurrent psychotropic polypharmacy, % [95%CI]"] 

polypharmacy_by_F01_F99_df = pd.DataFrame(columns = table2_column_names)
plot_df = pd.DataFrame(columns = ["ICD", "Total", "N1", "P1", "L1", "U1", "N2", "P2", "L2", "U2"])

lifetime_polypharmacy_df = cohort_polypharmacy_df[cohort_polypharmacy_df['lifetime_polypharmacy'] >= polypharmacy_threshold]
concurrent_polypharmacy_df = cohort_polypharmacy_df[cohort_polypharmacy_df['concurrent_polypharmacy'] >= polypharmacy_threshold]

polypharmacy_1_person_ids = set(lifetime_polypharmacy_df['person_id'].to_list())
polypharmacy_2_person_ids = set(concurrent_polypharmacy_df['person_id'].to_list())

F_icd_lst = ['F01_F99'] + cohort_icd_lst
for icd in F_icd_lst:
    pids = set(F_icd_code2pids[icd])
    diagnosis_no = len(pids)
    if diagnosis_no >= 322: # determined sample size by Scalex SP calculator
        num1 = len(set(pids & polypharmacy_1_person_ids))
        num2 = len(set(pids & polypharmacy_2_person_ids))
        p1, lower1, upper1 = None, None, None  
        if num1 > 0:
            p1, lower1, upper1 = calculate_percentage_ci(num1, diagnosis_no)
            item1 = f"{p1} [{lower1}, {upper1}]"
        else:
            item1 = "NA"
        p2, lower2, upper2 = None, None, None  
        if num2 > 0:
            p2, lower2, upper2 = calculate_percentage_ci(num2, diagnosis_no)
            item2 = f"{p2} [{lower2}, {upper2}]"
        else:
            item2 = "NA"

        polypharmacy_by_F01_F99_df.loc[len(polypharmacy_by_F01_F99_df)] = [f"{icd}: {icd_code2concept_name[icd]}", diagnosis_no, num1, item1, num2, item2]
        plot_df.loc[len(plot_df)] = [f"{icd}", diagnosis_no, num1, p1, lower1, upper1, num2, p2, lower2, upper2]

plot_df.to_excel(f"{output_dir}/results/tables/plot_Table_2_polypharmacy_by_F01_F99_diagnosis_{polypharmacy_threshold}_{consecutive_days}_cohort.xlsx", index=False)    
    
print(polypharmacy_by_F01_F99_df)    
polypharmacy_by_F01_F99_df.to_excel(f"{output_dir}/results/tables/Table_2_polypharmacy_by_F01_F99_diagnosis_{polypharmacy_threshold}_{consecutive_days}_cohort.xlsx", index=False)

In [ ]:
icd10cm_3_4_char_code2name_f = f'{output_dir}/data/all_4_char_ICD10CM_code2name.csv'
icd10cm_3_4_char_code2name_df = get_condition_icd_3_4_char_code_name(icd10cm_3_4_char_code2name_f)
icd10cm_3_4_char_code2name_df = pd.read_csv(icd10cm_3_4_char_code2name_f)
icd10cm_3_4_char_code2name_df.head()
print(len(icd10cm_3_4_char_code2name_df))

In [ ]:
# icd10cm_3_4_char_code2name_df['concept_code'].tolist()


In [ ]:
cohort_pids = cohort_polypharmacy_df['person_id'].tolist()
pid_charlson_df = charlson_index(cohort_pids, icd10cm_3_4_char_code2name_df, output_dir)

In [ ]:
pid_charlson_df['Multimorbidity'].value_counts()


In [ ]:
pid_charlson_df['CCI'].value_counts()


In [ ]:
cohort_polypharmacy_with_outcome_df = pd.merge(cohort_polypharmacy_df, pid_charlson_df[['person_id','Multimorbidity', 'CCI']],on='person_id', how='left')


In [ ]:
add_binary(cohort_polypharmacy_with_outcome_df, 'Multimorbidity', 'Multimorbidity_binary', multimorbidity_threshold) 
add_binary(cohort_polypharmacy_with_outcome_df, 'CCI', 'CCI_binary', multimorbidity_threshold)

In [ ]:
add_binary(cohort_polypharmacy_with_outcome_df, 'lifetime_polypharmacy', 'lifetime_polypharmacy_binary', polypharmacy_threshold)    
add_binary(cohort_polypharmacy_with_outcome_df, 'concurrent_polypharmacy', 'concurrent_polypharmacy_binary', polypharmacy_threshold)

In [ ]:
cohort_polypharmacy_with_outcome_df.columns


In [ ]:
import pandas as pd
from collections import Counter

f01_f99_pids = cohort_polypharmacy_with_outcome_df['person_id'].to_list()

icd2pids = {k: v for k, v in F_icd_code2pids.items() if k != 'F01_F99'}

all_pids = []
for pids in icd2pids.values():
    all_pids.extend(pids)

pid_counter = Counter(all_pids)

psychiatric_multimorbidity_df = pd.DataFrame(
    {'person_id': f01_f99_pids, 
     'psychiatric_multimorbidity': [pid_counter.get(pid, 0) for pid in f01_f99_pids]}
)

In [ ]:
psychiatric_multimorbidity_df["psychiatric_multimorbidity"].value_counts()


In [ ]:
cohort_polypharmacy_with_outcome_df = pd.merge(cohort_polypharmacy_with_outcome_df, psychiatric_multimorbidity_df,on='person_id', how='left')


In [ ]:
# lifetime polypharmacy
lifetime_polypharmacy_df = cohort_polypharmacy_with_outcome_df[cohort_polypharmacy_with_outcome_df['lifetime_polypharmacy_binary']==1]
lifetime_polypharmacy_df.to_csv(f"{output_dir}/results/tables/lifetime_polypharmacy_{polypharmacy_threshold}_{consecutive_days}.csv", index=False)
print(len(lifetime_polypharmacy_df))
lifetime_polypharmacy_df['psychiatric_multimorbidity'].value_counts()

In [ ]:
concurrent_polypharmacy_df = cohort_polypharmacy_with_outcome_df[cohort_polypharmacy_with_outcome_df['concurrent_polypharmacy_binary']==1]
concurrent_polypharmacy_df.to_csv(f"{output_dir}/results/tables/concurrent_polypharmacy_{polypharmacy_threshold}_{consecutive_days}.csv", index=False)
print(len(concurrent_polypharmacy_df))
concurrent_polypharmacy_df['psychiatric_multimorbidity'].value_counts()

In [ ]:
concurrent_polypharmacy_df['psychiatric_diagnoses']


In [ ]:
concurrent_polypharmacy_df[concurrent_polypharmacy_df['psychiatric_diagnoses']=={'F20'}]


In [ ]:
concurrent_polypharmacy_df['psychiatric_diagnoses'].value_counts()


In [ ]:
output_file = f"{output_dir}/results/tables/upset_data_lifetime.csv"

lifetime_upset_df = generate_upset_data(lifetime_polypharmacy_df,output_file)


In [ ]:
concurrent_output_file = f"{output_dir}/results/tables/upset_data_concurrent.csv"

concurrent_upset_df = generate_upset_data(concurrent_polypharmacy_df,concurrent_output_file)


In [ ]:
len(cohort_polypharmacy_with_outcome_df)


In [ ]:
target_df = cohort_polypharmacy_with_outcome_df
output_f = f'{output_dir}/results/tables/Table_top_atc_lifetime_polypharmacy_{polypharmacy_threshold}_{consecutive_days}_statistics.xlsx'
target_column='lifetime_polypharmacy'
# top_n=20
top_n = None
top_atc_lifetime_polypharmcy_df = medication_distribution(atc_N_code2name_df, target_df, TOTAL_count, output_f, polypharmacy_threshold, target_column, top_n)

In [ ]:
top_atc_lifetime_polypharmcy_df

In [ ]:
cohort_polypharmacy_with_outcome_df["concurrent_polypharmacy"].value_counts()

In [ ]:
list_of_sets = cohort_polypharmacy_with_outcome_df["concurrent_polypharmacy_atc_codes"].tolist()
# print(list_of_sets)
frozen_sets = [frozenset(ss) for ss in list_of_sets]

# Count occurrences
counts = Counter(frozen_sets)
print(len(counts))

In [ ]:
# target_df = cohort_polypharmacy_with_outcome_df.copy()
# target_column2 = "concurrent_polypharmacy_atc_codes"

# print(f"Has missing values: {target_df[target_column2].isna().any()}")
# print(f"Number of missing values: {target_df[target_column2].isna().sum()}")
# print(f"Percentage missing: {target_df[target_column2].isna().mean() * 100:.2f}%")

In [ ]:
target_column2 = "concurrent_polypharmacy_atc_codes"
target_column = 'concurrent_polypharmacy'

selected_columns = ["person_id", 'concurrent_polypharmacy', "concurrent_polypharmacy_atc_codes"]

polypharmacy_df = target_df[target_df[target_column] >= 4][selected_columns].copy()
#########################

# list_of_sets = polypharmacy_df[target_column2].tolist()

# frozen_sets = [frozenset(ss) for ss in list_of_sets]

# # Count occurrences
# counts = Counter(frozen_sets)
# print(len(counts))

##########################
total = len(polypharmacy_df)
print(total)
all_medication_lst = []
for index, row in polypharmacy_df.iterrows():
    flattened_list = row[target_column2]
    # print(flattened_list)
    all_medication_lst.extend(flattened_list)

counter = Counter(all_medication_lst)

tops = counter.most_common(top_n)
print(tops)
result_df = pd.DataFrame(columns = ["Rank", "Medication (ATC 5th)", f"Participants, % (N={total})", "Class (ATC 3rd level)"])
rank = 1
for (atc_code, count) in tops:
    atc_name = atc2name(atc_N_code2name_df, atc_code)
    ptg_str = "{:.2f}".format(count/total * 100)
    
    atc_3 = atc_code[:4]
    atc_3_name = atc2name(atc_N_code2name_df, atc_3)
    
    row = [rank, f"{atc_name} ({atc_code})",ptg_str, f"{atc_3_name} ({atc_3})"]
    rank += 1
    result_df.loc[len(result_df)] = row

In [ ]:
target_df = cohort_polypharmacy_with_outcome_df
output_f = f'{output_dir}/results/tables/Table_top_atc_concurrent_polypharmacy_{polypharmacy_threshold}_{consecutive_days}_statistics.xlsx'
target_column='concurrent_polypharmacy'
# top_n=20
top_n = None
top_atc_concurrent_polypharmcy_df = medication_distribution(atc_N_code2name_df, target_df, TOTAL_count, output_f, polypharmacy_threshold, target_column, top_n)

In [ ]:
top_atc_concurrent_polypharmcy_df

In [ ]:
cohort_polypharmacy_with_outcome_df[cohort_polypharmacy_with_outcome_df['person_id'] == 3291352]

In [ ]:
# sorted(cohort_polypharmacy_with_outcome_df["concurrent_polypharmacy_atc_codes"], reverse=True)

In [ ]:
target_df = cohort_polypharmacy_with_outcome_df
output_f = f'{output_dir}/results/tables/Table_top_atc_concurrent_polypharmacy_{polypharmacy_threshold}_{consecutive_days}_statistics.xlsx'
target_column='concurrent_polypharmacy'
# top_n=20
top_n = None
top_atc_concurrent_polypharmcy_df = medication_distribution(atc_N_code2name_df, target_df, TOTAL_count, output_f, polypharmacy_threshold, target_column, top_n)

In [ ]:
top_atc_concurrent_polypharmcy_df

In [ ]:
top_atc_concurrent_polypharmcy_df

In [ ]:
len(top_atc_concurrent_polypharmcy_df)


In [ ]:
len(top_atc_lifetime_polypharmcy_df)


In [ ]:
cohort_pids = cohort_polypharmacy_df['person_id'].tolist()

visit_output_f = f"{output_dir}/data/cohort_visit_occurrence.csv"
hospitalisation_df = get_hospitalisation(cohort_pids,visit_output_f, save_data=True)
visit_occurrence_df = pd.read_csv(visit_output_f)

In [ ]:
visit_occurrence_df

In [ ]:
len(cohort_polypharmacy_with_outcome_df)


In [ ]:
## get visit count
print(visit_occurrence_df.isna().sum())
print(len(visit_occurrence_df))

## merge data
cohort_polypharmacy_with_outcome_ERV_IV_count_df = pd.merge(
    cohort_polypharmacy_with_outcome_df, 
    visit_occurrence_df[['person_id', "concept_name", 'visit_start_date']], 
    on='person_id', 
    how='left'  
)

n_jobs = 8
ERV_IV_output = f"{output_dir}/data/ERV_IV_visit_count.csv"
ERV_IV_visit_count_df = get_visit_count(cohort_polypharmacy_with_outcome_ERV_IV_count_df, n_jobs, ERV_IV_output, True)
ERV_IV_visit_count_df = pd.read_csv(ERV_IV_output)

In [ ]:
cohort_polypharmacy_with_all_outcome_df = pd.merge(cohort_polypharmacy_with_outcome_df, ERV_IV_visit_count_df[['person_id', 'ERV_count', 'IV_count']],on='person_id', how='left')


In [ ]:
cohort_polypharmacy_with_all_outcome_df.columns


In [ ]:
add_binary(cohort_polypharmacy_with_all_outcome_df, 'ERV_count', 'ERV_binary', ERV_threshold) 
add_binary(cohort_polypharmacy_with_all_outcome_df, 'IV_count', 'IV_binary', IV_threshold)

In [ ]:
cohort_polypharmacy_with_all_outcome_df['death'] = np.where(cohort_polypharmacy_with_all_outcome_df['death_date'].notna(), 1, 0)
print(cohort_polypharmacy_with_all_outcome_df.isna().sum())
print(len(cohort_polypharmacy_with_all_outcome_df))

In [ ]:
cohort_polypharmacy_with_all_outcome_df['death'].value_counts()


In [ ]:
def death_group(val):
    if val == 1:
        return '1'
    else:
        return '0'

In [ ]:
cohort_polypharmacy_with_all_outcome_df['death_group'] = cohort_polypharmacy_with_all_outcome_df['death'].apply(death_group)
cohort_polypharmacy_with_all_outcome_df['death_group'].value_counts()

In [ ]:
def multimorbidity_group(val):
    if val == 1:
        return '1'
    elif (val >= 2 and val <= 4):
        return '2-4'
    elif (val >= 5 and val <= 7):
        return '5-7'
    elif (val >= 8):
        return '>=8'
    else:
        return '0'

In [ ]:
cohort_polypharmacy_with_all_outcome_df['Multimorbidity_group'] = cohort_polypharmacy_with_all_outcome_df['Multimorbidity'].apply(multimorbidity_group)
cohort_polypharmacy_with_all_outcome_df['Multimorbidity_group'].value_counts()

In [ ]:
def CCI_score_group(val):
    if (val >= 1 and val <= 2):
        return '1-2'
    elif (val >= 3 and val <= 4):
        return '3-4'
    elif (val >=5):
        return '>=5'
    else:
        return '0'

In [ ]:
cohort_polypharmacy_with_all_outcome_df['CCI_group'] = cohort_polypharmacy_with_all_outcome_df['CCI'].apply(CCI_score_group)
cohort_polypharmacy_with_all_outcome_df['CCI_group'].value_counts()

In [ ]:
def ERV_IV_count_group(val):
    if (val >= 1 and val <= 5):
        return '1-5'   
    elif (val >= 6 and val <= 10):
        return '6-10'
    elif (val >= 11 and val <= 20):
        return '11-20'   
    elif (val >= 21):
        return '>=21'    
    else:
        return '0'

In [ ]:
cohort_polypharmacy_with_all_outcome_df['ERV_count_group'] = cohort_polypharmacy_with_all_outcome_df['ERV_count'].apply(ERV_IV_count_group)
cohort_polypharmacy_with_all_outcome_df['IV_count_group'] = cohort_polypharmacy_with_all_outcome_df['IV_count'].apply(ERV_IV_count_group)

In [ ]:
cohort_polypharmacy_with_all_outcome_df['ERV_count_group'].value_counts()


In [ ]:
cohort_polypharmacy_with_all_outcome_df['IV_count_group'].value_counts()


In [ ]:
multimorbidity_df = cohort_polypharmacy_with_all_outcome_df[cohort_polypharmacy_with_all_outcome_df["Multimorbidity"] >= multimorbidity_threshold]
multimorbidity_value_counts = multimorbidity_df['Multimorbidity_group'].value_counts()

In [ ]:
cohort_polypharmacy_with_all_outcome_df.columns


In [ ]:
import pandas as pd

def adverse_event_rates_by_polypharmacy(df):
    # Group by lifetime_polypharmacy
    lifetime_summary = df.groupby('lifetime_polypharmacy').agg(
        mean_ERV_count=('ERV_count', 'mean'),
        rate_ERV_binary=('ERV_binary', 'mean'),
        mean_IV_count=('IV_count', 'mean'),
        rate_IV_binary=('IV_binary', 'mean'),
        mean_CCI=('CCI', 'mean'),
        mean_Multimorbidity=('Multimorbidity', 'mean')
    ).reset_index()

    # Group by concurrent_polypharmacy
    concurrent_summary = df.groupby('concurrent_polypharmacy').agg(
        mean_ERV_count=('ERV_count', 'mean'),
        rate_ERV_binary=('ERV_binary', 'mean'),
        mean_IV_count=('IV_count', 'mean'),
        rate_IV_binary=('IV_binary', 'mean'),
        mean_CCI=('CCI', 'mean'),
        mean_Multimorbidity=('Multimorbidity', 'mean')
    ).reset_index()

    # Convert binary rates to percentages
    for summary in [lifetime_summary, concurrent_summary]:
        summary['rate_ERV_binary'] = summary['rate_ERV_binary'] * 100
        summary['rate_IV_binary'] = summary['rate_IV_binary'] * 100

    return lifetime_summary, concurrent_summary

In [ ]:
lifetime_rates, concurrent_rates = adverse_event_rates_by_polypharmacy(cohort_polypharmacy_with_all_outcome_df)
print(lifetime_rates)
print(concurrent_rates)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def create_summary(df, polypharmacy_col):
    # Group and aggregate mean for your variables of interest
    agg_cols = ['ERV_count', 'IV_count', 'CCI', 'Multimorbidity']  # add any other columns you want to plot
    
    summary = df.groupby(polypharmacy_col)[agg_cols].mean().reset_index()
    
    return summary

def preprocess_polypharmacy_summary(df, polypharmacy_col):
    df = df.copy()
    # Convert to numeric (int), coerce errors to NaN then fill or drop as needed
    df[polypharmacy_col] = pd.to_numeric(df[polypharmacy_col], errors='coerce').fillna(0).astype(int)
    
    # Group all values >=10 as 10
    df[polypharmacy_col] = df[polypharmacy_col].apply(lambda x: x if x < 10 else 10)

    # Aggregate again in case grouping put multiple rows to 10
    agg_cols = [col for col in df.columns if col != polypharmacy_col]
    df = df.groupby(polypharmacy_col)[agg_cols].mean().reset_index()

    # Convert polypharmacy_col to string for '10+' label
    df[polypharmacy_col] = df[polypharmacy_col].astype(str)
    df.loc[df[polypharmacy_col] == '10', polypharmacy_col] = '10+'

    all_groups = [str(i) for i in range(10)] + ['10+']

    df = df.set_index(polypharmacy_col).reindex(all_groups, fill_value=0).reset_index()

    return df

def plot_polypharmacy_vs_visits_and_cci(lifetime_summary, concurrent_summary):
    plt.figure(figsize=(12,6))
    
    # Lifetime polypharmacy
    plt.plot(
        lifetime_summary['lifetime_polypharmacy'], 
        lifetime_summary['ERV_count'], 
        marker='o', linestyle='-', color='blue', label='Lifetime Polypharmacy vs ERV'
    )
    plt.plot(
        lifetime_summary['lifetime_polypharmacy'], 
        lifetime_summary['IV_count'], 
        marker='o', linestyle='--', color='blue', label='Lifetime Polypharmacy vs IV'
    )
    plt.plot(
        lifetime_summary['lifetime_polypharmacy'], 
        lifetime_summary['CCI'], 
        marker='o', linestyle=':', color='blue', label='Lifetime Polypharmacy vs CCI'
    )
    
    # Concurrent polypharmacy
    plt.plot(
        concurrent_summary['concurrent_polypharmacy'], 
        concurrent_summary['ERV_count'], 
        marker='s', linestyle='-', color='red', label='Concurrent Polypharmacy vs ERV'
    )
    plt.plot(
        concurrent_summary['concurrent_polypharmacy'], 
        concurrent_summary['IV_count'], 
        marker='s', linestyle='--', color='red', label='Concurrent Polypharmacy vs IV'
    )
    plt.plot(
        concurrent_summary['concurrent_polypharmacy'], 
        concurrent_summary['CCI'], 
        marker='s', linestyle=':', color='red', label='Concurrent Polypharmacy vs CCI'
    )
    
    plt.xlabel('Polypharmacy Number')
    plt.ylabel('Mean Counts / Score')
    plt.title('Mean ERV, IV Visit Counts and CCI Score by Polypharmacy Level')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
print(cohort_polypharmacy_with_all_outcome_df['lifetime_polypharmacy'].duplicated().any())
print(cohort_polypharmacy_with_all_outcome_df['concurrent_polypharmacy'].duplicated().any())

In [ ]:
len(set((cohort_polypharmacy_with_all_outcome_df['person_id'].tolist())))


In [ ]:
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt

def plot_polypharmacy_visits_and_cci_concurrent_only(concurrent_summary):

    fig = plt.figure(figsize=(150/25.4, 100/25.4)) #plt.figure(figsize=(12, 6))
    gs = GridSpec(3, 1, height_ratios=[1, 2, 0.05], figure=fig)
    
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1], sharex=ax1)

    x_conc = concurrent_summary['concurrent_polypharmacy']

    # --- Top subplot: CCI score ---
    ax1.plot(
        x_conc, concurrent_summary['CCI'],
        marker='^', linestyle='-', color='red',markersize=4, linewidth=1
    )

    ax1.set_xlabel('Number of medications', fontsize=7)
    ax1.set_ylabel('Mean CCI score', fontsize=7)
    ax1.grid(False)
    ax1.set_xlim(ax1.get_xlim())
    ax1.set_ylim(0, 5)
    ax1.set_yticks(np.arange(0, 6, 1))
    ax1.tick_params(axis='both', labelsize=6)
        
    # --- Bottom subplot: ERV and IV counts ---
    ax2.plot(
        x_conc, concurrent_summary['ERV_count'],
        marker='o', linestyle='-', color='red',markersize=4, linewidth=1
    )
    ax2.plot(
        x_conc, concurrent_summary['IV_count'],
        marker='s', linestyle='-', color='red',markersize=4, linewidth=1
    )

    ax2.set_ylabel('Mean times of visits', fontsize=7)
    ax2.set_xlabel('Number of medications', fontsize=7)
    ax2.grid(False)
    ax2.tick_params(axis='both', labelsize=6)

    
    top_legend_handles = [
        Line2D([0], [0], color='red', marker='^', linestyle='None', label='CCI')
    ]
    ax1.legend(
        handles=top_legend_handles, 
        loc='upper left', 
        frameon=False,
        fontsize=5.5,
        title_fontsize=6,
        markerscale=0.7,
        # handlelength=1.2,
        # handletextpad=0.4,
        # borderpad=0.1,
        # labelspacing=0.5
    )

    bottom_legend_handles = [
        Line2D([0], [0], color='red', marker='o', linestyle='None', label='ER visits'),
        Line2D([0], [0], color='red', marker='s', linestyle='None', label='Inpatient visits'),
    ]
    
    ax2.legend(
        handles=bottom_legend_handles, 
        loc='upper left', 
        frameon=False,
        fontsize=5.5,
        title_fontsize=6,
        markerscale=0.7,
        # handlelength=1.2,
        # handletextpad=0.4,
        # borderpad=0.1,
        # labelspacing=0.5
    )
    
    plt.tight_layout(rect=[0, 0, 1, 0.97])

    plt.savefig('/tmp/concurrent_polypharmacy_vs_visit_cci.pdf', bbox_inches='tight')
    plt.savefig('/tmp/concurrent_polypharmacy_vs_visit_cci.svg', bbox_inches='tight')

    for ext in ["pdf", "svg"]:
        local_path = f"/tmp/concurrent_polypharmacy_vs_visit_cci.{ext}"
        plt.savefig(local_path, bbox_inches="tight")

        blob = bucket.blob(f"results/figures/concurrent_polypharmacy_vs_visit_cci.{ext}")
        blob.upload_from_filename(local_path)
    
    
    plt.show()

In [ ]:
lifetime_summary = create_summary(cohort_polypharmacy_with_all_outcome_df, 'lifetime_polypharmacy')
concurrent_summary = create_summary(cohort_polypharmacy_with_all_outcome_df, 'concurrent_polypharmacy')

lifetime_df = preprocess_polypharmacy_summary(lifetime_summary, 'lifetime_polypharmacy')
concurrent_df = preprocess_polypharmacy_summary(concurrent_summary, 'concurrent_polypharmacy')

plot_polypharmacy_visits_and_cci_concurrent_only(concurrent_df)

In [ ]:
##################################################################
# polypharmacy by outcomes
## Table 3 psychotropic polypharmacy by multimorbidity, cci, emergency room visit and inpatient visit

target_df = cohort_polypharmacy_with_all_outcome_df

table2_column_names = ["Outcome", 
                     "Nnumber of participants", 
                     "Lifetime psychotropic polypharmacy, N",
                     "Lifetime psychotropic polypharmacy, % (95%CI)",
                     "Concurrent psychotropic polypharmacy, N",
                     "Concurrent psychotropic polypharmacy, % (95%CI)"] 

polypharmacy_by_outcome_df = pd.DataFrame(columns = table2_column_names)
#######################################################
lifetime_polypharmacy_df = target_df[target_df['lifetime_polypharmacy'] >= polypharmacy_threshold]
lifetime_concurrent_polypharmacy_df = target_df[target_df['concurrent_polypharmacy'] >= polypharmacy_threshold]

medication_exposure_df = target_df.dropna(subset=['lifetime_polypharmacy'])
medication_exposure_pids = set(medication_exposure_df['person_id'].to_list())
print(len(medication_exposure_pids))

polypharmacy_1_person_ids = set(lifetime_polypharmacy_df['person_id'].to_list())
polypharmacy_2_person_ids = set(lifetime_concurrent_polypharmacy_df['person_id'].to_list())
#######################################################
# multimorbidity_df = {}
# multimorbidity_values = ['No', 'Yes', 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
# for val in multimorbidity_values:
#     if val == 'Yes': # medication yes
#         multimorbidity_df[val] = target_df[target_df[target_column] > 0]
#     elif val == 'No': # condition no
#         multimorbidity_df[val] = target_df[target_df[target_column] == 0]
#     else:
#         multimorbidity_df[val] = target_df[target_df[target_column] == val]
#         # print(multimorbidity_df[val].isna().sum())

# multimorbidity_df['>10'] = target_df[target_df[target_column] > 10]
multimorbidity_df = target_df[target_df["Multimorbidity"] >= multimorbidity_threshold]

CCI_df = target_df[target_df["CCI"] >= 0]

#######################################################
ERV_df = target_df[target_df["ERV_count"] >= 0]
IV_df = target_df[target_df["IV_count"] >=0 ]

#########################################################
# multimorbidity_group_df = {f'Multimorbidity (>={multimorbidity_threshold} diagnosis)': multimorbidity_df,
#               '2-4': multimorbidity_df[multimorbidity_df['Multimorbidity_group']=='2-4'],
#               '5-7': multimorbidity_df[multimorbidity_df['Multimorbidity_group']=='5-7'],
#               '>=8': multimorbidity_df[multimorbidity_df['Multimorbidity_group']=='>=8']}

# CCI_group_df = {f'CCI score': CCI_df, 
#                 '0': CCI_df[CCI_df['CCI_group']=='0'],
#                 '1': CCI_df[CCI_df['CCI_group']=='1'],
#               '2-5': CCI_df[CCI_df['CCI_group']=='2-5'],
#               '6-9': CCI_df[CCI_df['CCI_group']=='6-9'],
#               '>=10': CCI_df[CCI_df['CCI_group']=='>=10']}

CCI_group_df = {f'CCI score': CCI_df, 
                '0': CCI_df[CCI_df['CCI_group']=='0'],
                '1-2': CCI_df[CCI_df['CCI_group']=='1-2'],
              '3-4': CCI_df[CCI_df['CCI_group']=='3-4'],
              '>=5': CCI_df[CCI_df['CCI_group']=='>=5']}

# ERV_group_df = {"Emergency room visit": ERV_df, 
#                 '0': ERV_df[ERV_df['ERV_count_group']=='0'],
#               '1-5': ERV_df[ERV_df['ERV_count_group']=='1-5'],
#               '6-10': ERV_df[ERV_df['ERV_count_group']=='6-10'],
#               '11-20': ERV_df[ERV_df['ERV_count_group']=='11-20'],
#               '21-30': ERV_df[ERV_df['ERV_count_group']=='21-30'],
#               '31-40': ERV_df[ERV_df['ERV_count_group']=='31-40'], 
#               '>=41': ERV_df[ERV_df['ERV_count_group']=='>=41']}

# IV_group_df = {"Inpatient visit": IV_df,
#                '0': IV_df[IV_df['IV_count_group']=='0'],
#               '1-5': IV_df[IV_df['IV_count_group']=='1-5'],
#               '6-10': IV_df[IV_df['IV_count_group']=='6-10'],
#               '11-20': IV_df[IV_df['IV_count_group']=='11-20'],
#               '21-30': IV_df[IV_df['IV_count_group']=='21-30'],
#               '31-40': IV_df[IV_df['IV_count_group']=='31-40'], 
#               '>=41': IV_df[IV_df['IV_count_group']=='>=41']            
#              }

ERV_group_df = {"Emergency room visit": ERV_df, 
                '0': ERV_df[ERV_df['ERV_count_group']=='0'],
              '1-5': ERV_df[ERV_df['ERV_count_group']=='1-5'],
              '6-10': ERV_df[ERV_df['ERV_count_group']=='6-10'],
              '11-20': ERV_df[ERV_df['ERV_count_group']=='11-20'],
              '>=21': ERV_df[ERV_df['ERV_count_group']=='>=21']}

IV_group_df = {"Inpatient visit": IV_df,
               '0': IV_df[IV_df['IV_count_group']=='0'],
              '1-5': IV_df[IV_df['IV_count_group']=='1-5'],
              '6-10': IV_df[IV_df['IV_count_group']=='6-10'],
              '11-20': IV_df[IV_df['IV_count_group']=='11-20'],
              '>=21': IV_df[IV_df['IV_count_group']=='>=21']            
             }

outcome_groups = [CCI_group_df, ERV_group_df, IV_group_df] #multimorbidity_group_df, 

for group in outcome_groups:
    for outcome, cur_df in group.items():

        cur_pids = set(cur_df['person_id'].tolist())
        outcome_no = len(cur_pids)

        meidcation_num = len(set(medication_exposure_pids & cur_pids))

        if meidcation_num > 0 and outcome_no > 0:
            item_medication = calculate_percentage_ci(meidcation_num, outcome_no)
        else:
            item_medication = "NA"

        num1 = len(set(cur_pids & polypharmacy_1_person_ids))
        num2 = len(set(cur_pids & polypharmacy_2_person_ids))

        # poly_p1 = "{:.2f}".format(num1/TOTAL_count * 100)
        # poly_p2 = "{:.2f}".format(num2/TOTAL_count * 100)
        #if diagnosis_no > 0:
        #    item1 = calculate_percentage_ci(diagnosis_no, TOTAL_count)
        #else:
        #    item1 = "NA"
        if meidcation_num > 0 and num1 > 0:
            p2, l2, u2 = calculate_percentage_ci(num1, outcome_no)
            item2 = f"{p2} [{l2}, {u2}]"
        else:
            item2 = "NA"
        if meidcation_num > 0 and num2 > 0:
            p3, l3, u3 = calculate_percentage_ci(num2, outcome_no)
            item3 = f"{p3} [{l3}, {u3}]"
        else:
            item3 = "NA"

        polypharmacy_by_outcome_df.loc[len(polypharmacy_by_outcome_df)] = [outcome, outcome_no, num1, item2, num2, item3]

print(polypharmacy_by_outcome_df)    
polypharmacy_by_outcome_df.to_excel(f"{output_dir}/results/tables/Table_3_polypharmacy_by_outcomes_{polypharmacy_threshold}_{consecutive_days}_cohort_EHR.xlsx", index=False)

In [ ]:
cohort_polypharmacy_with_all_outcome_df["log_exposure"] = np.log1p(cohort_polypharmacy_with_all_outcome_df["duration"])

cohort_polypharmacy_with_all_outcome_df["log_exposure_std"] = (
    cohort_polypharmacy_with_all_outcome_df["log_exposure"] - cohort_polypharmacy_with_all_outcome_df["log_exposure"].mean()
) / cohort_polypharmacy_with_all_outcome_df["log_exposure"].std()

In [ ]:
cohort_polypharmacy_with_all_outcome_df['log_exposure_std']


In [ ]:
set(cohort_polypharmacy_with_all_outcome_df['psychiatric_diagnoses_count'].tolist())


In [ ]:
##################################################################
## Table 4 psychotropic polypharmacy by the number of diagnosis

target_df = cohort_polypharmacy_with_all_outcome_df

table2_column_names = ["Diagnosis number", 
                     "Nnumber of participants", 
                     "Lifetime psychotropic polypharmacy, No.",
                     "Lifetime psychotropic polypharmacy, % (95% CI)",
                     "Concurrent psychotropic polypharmacy, No.",
                     "Concurrent psychotropic polypharmacy, % (95% CI)"] 

polypharmacy_by_outcome_df = pd.DataFrame(columns = table2_column_names)
plot_df = pd.DataFrame(columns = ["DiagnosisNum", "Total", "N1", "P1", "L1", "U1", "N2", "P2", "L2", "U2"])


#######################################################
lifetime_polypharmacy_df = target_df[target_df['lifetime_polypharmacy'] >= polypharmacy_threshold]
concurrent_polypharmacy_df = target_df[target_df['concurrent_polypharmacy'] >= polypharmacy_threshold]

medication_exposure_df = target_df.dropna(subset=['lifetime_polypharmacy'])
medication_exposure_pids = set(medication_exposure_df['person_id'].to_list())
print(len(medication_exposure_pids))

polypharmacy_1_person_ids = set(lifetime_polypharmacy_df['person_id'].to_list())
polypharmacy_2_person_ids = set(concurrent_polypharmacy_df['person_id'].to_list())
#######################################################

#diganosis_nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, ">=20"]
diganosis_nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, ">=15"]

# diganosis_nums = list(set(cohort_polypharmacy_with_all_outcome_df['psychiatric_diagnoses_count'].tolist()))


for diagnosis_num in diganosis_nums:
    
    if diagnosis_num == ">=15":
        cur_df = cohort_polypharmacy_with_all_outcome_df[cohort_polypharmacy_with_all_outcome_df['psychiatric_diagnoses_count']>=15]
    else:
        cur_df = cohort_polypharmacy_with_all_outcome_df[cohort_polypharmacy_with_all_outcome_df['psychiatric_diagnoses_count']==diagnosis_num]
    
    cur_pids = set(cur_df['person_id'].tolist())
    cur_diagnosis = len(cur_pids)

    num1 = len(set(cur_pids & polypharmacy_1_person_ids))
    num2 = len(set(cur_pids & polypharmacy_2_person_ids))

    if cur_diagnosis > 0 and num1 > 0:
        p1, l1, u1 = calculate_percentage_ci(num1, cur_diagnosis)
        item1 = f"{p1} [{l1}, {u1}]"
    else:
        item1 = "NA"
    if cur_diagnosis > 0 and num2 > 0:
        p2, l2, u2 = calculate_percentage_ci(num2, cur_diagnosis)
        item2 = f"{p2} [{l2}, {u2}]"
    else:
        item2 = "NA"

    polypharmacy_by_outcome_df.loc[len(polypharmacy_by_outcome_df)] = [diagnosis_num, cur_diagnosis, num1, item2, num2, item3]
    plot_df.loc[len(plot_df)] = [diagnosis_num, cur_diagnosis, num1, p1, l1, u1, num2, p2, l2, u2]

plot_df.to_excel(f"{output_dir}/results/tables/plot_Table_4_polypharmacy_by_num_of_diagnosis_{polypharmacy_threshold}_{consecutive_days}_cohort.xlsx", index=False)    
  
print(plot_df)    
polypharmacy_by_outcome_df.to_excel(f"{output_dir}/results/tables/Table_4_polypharmacy_by_num_of_diagnosis_{polypharmacy_threshold}_{consecutive_days}_cohort.xlsx", index=False)

In [ ]:
##################################################################
## Table 5 psychotropic polypharmacy by top ranking co-diagnosis
#target_df = cohort_polypharmacy_with_all_outcome_df
def polypharmacy_by_top_co_diagnosis(target_df,co_diagnosis_n=2,top_comb_n=20):
    # top_comb_n = 20
    # co_diagnosis_n = 2

    multi_diagnosis_df = target_df[target_df['psychiatric_diagnoses_count']>=co_diagnosis_n]    

    top_diagnosis_comb_list = multi_diagnosis_df['psychiatric_diagnoses'].value_counts().head(top_comb_n).index.tolist()    

    filtered_df = target_df[target_df['psychiatric_diagnoses'].isin(top_diagnosis_comb_list)]

    top_com2count = filtered_df['psychiatric_diagnoses'].value_counts()


    pids = filtered_df['person_id'].tolist()
    top_diagnosis_icds = set([item for sublist in top_diagnosis_comb_list for item in sublist])
    top_diagnosis_icds


    table2_column_names = ["CoDiagnosis", 
                         "Nnumber of participants", 
                         "Lifetime psychotropic polypharmacy, N",
                         "Lifetime psychotropic polypharmacy, % (95%CI)",
                         "Concurrent psychotropic polypharmacy, N",
                         "Concurrent psychotropic polypharmacy, % (95%CI)"] 

    polypharmacy_by_outcome_df = pd.DataFrame(columns = table2_column_names)
    plot_df = pd.DataFrame(columns = ["CoDiagnosis", "Total", "N1", "P1", "L1", "U1", "N2", "P2", "L2", "U2"])


    # #######################################################
    lifetime_polypharmacy_df = target_df[target_df['lifetime_polypharmacy'] >= polypharmacy_threshold]
    concurrent_polypharmacy_df = target_df[target_df['concurrent_polypharmacy'] >= polypharmacy_threshold]

    polypharmacy_1_person_ids = set(lifetime_polypharmacy_df['person_id'].to_list())
    polypharmacy_2_person_ids = set(concurrent_polypharmacy_df['person_id'].to_list())
    # #######################################################

    diganosis_nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

    for co_diagnosis, count in top_com2count.items():
        print(co_diagnosis, count)

        cur_df = target_df[target_df['psychiatric_diagnoses']==co_diagnosis]
        #print(cur_df)

        cur_pids = set(cur_df['person_id'].tolist())
        cur_diagnosis = len(cur_pids)

        num1 = len(set(cur_pids & polypharmacy_1_person_ids))
        num2 = len(set(cur_pids & polypharmacy_2_person_ids))

        if cur_diagnosis > 0 and num1 > 0:
            p1, l1, u1 = calculate_percentage_ci(num1, cur_diagnosis)
            item1 = f"{p1} [{l1}, {u1}]"
        else:
            item1 = "NA"
        if cur_diagnosis > 0 and num2 > 0:
            p2, l2, u2 = calculate_percentage_ci(num2, cur_diagnosis)
            item2 = f"{p2} [{l2}, {u2}]"
        else:
            item2 = "NA"

        polypharmacy_by_outcome_df.loc[len(polypharmacy_by_outcome_df)] = [co_diagnosis, cur_diagnosis, num1, item2, num2, item3]
        plot_df.loc[len(plot_df)] = [co_diagnosis, cur_diagnosis, num1, p1, l1, u1, num2, p2, l2, u2]

    print(plot_df['CoDiagnosis'])
        
    plot_df['SetSize'] = plot_df['CoDiagnosis'].astype(str).str.strip('{}').str.split(',').apply(len)
    plot_df_sorted = plot_df.sort_values(by=['SetSize', 'Total'], ascending=[True, False]).drop(columns='SetSize')        
        
    plot_df_sorted.to_excel(f"{output_dir}/results/tables/plot_Table_5_polypharmacy_by_top_20_co_diagnosis_{polypharmacy_threshold}_{consecutive_days}_cohort.xlsx", index=False)    

    print(plot_df)  
    print(plot_df_sorted)
    polypharmacy_by_outcome_df.to_excel(f"{output_dir}/results/tables/Table_5_polypharmacy_by_top_20_co_diagnosis_{polypharmacy_threshold}_{consecutive_days}_cohort.xlsx", index=False)
    
    return polypharmacy_by_outcome_df
polypharmacy_by_co_diagnosis_df = polypharmacy_by_top_co_diagnosis(target_df)

In [ ]:

# polypharmacy by multimorbidity, co-diagnosis of mental health disorders
def generate_upset_data_base(co_diagnosis_lst, target_df):
    
    filtered_df = target_df[target_df['psychiatric_diagnoses'].isin(co_diagnosis_lst)]
        
    pids = filtered_df['person_id'].tolist()
    top_diagnosis_icds = set([item for sublist in co_diagnosis_lst for item in sublist])
    
    unique_icds = set()
    for s in co_diagnosis_lst:
        #icd = ast.literal_eval(s)
        unique_icds.update(s)
    print(unique_icds)
    
    unique_icd_lst = list(unique_icds)
    upset_df = pd.DataFrame(False, index=sorted(pids), columns=sorted(unique_icd_lst))
    
    #print(filtered_df['psychiatric_diagnoses'])
    
    for pid in pids:
        #print(pid)
        #icd_lst = ast.literal_eval(filtered_df[filtered_df['person_id']==pid]['psychiatric_diagnoses'].tolist()[0])
        icd_lst = filtered_df[filtered_df['person_id'] == pid]['psychiatric_diagnoses'].tolist()[0]

        for icd in icd_lst:         
            upset_df.loc[pid, icd] = True

    print(upset_df)
    print(len(upset_df))
    upset_df = upset_df.loc[~(upset_df.eq(False).all(axis=1))]

    upset_df = upset_df[upset_df.any(axis=1)]
    upset_df = upset_df.astype(int)    

    upset_df.to_csv(f"{output_dir}/results/tables/upset_data_combinations_for_figure4_{polypharmacy_threshold}.csv")
    print(upset_df)
    return upset_df




In [ ]:
co_diagnosis_lst = polypharmacy_by_co_diagnosis_df['CoDiagnosis'].to_list()

target_df = cohort_polypharmacy_with_all_outcome_df

generate_upset_data_base(co_diagnosis_lst, target_df)

In [ ]:
cohort_polypharmacy_with_all_outcome_df.columns

In [ ]:
cohort_polypharmacy_with_all_outcome_df["psychiatric_diagnoses"]

In [ ]:
cohort_polypharmacy_with_all_outcome_df['death'].value_counts()


In [ ]:
cohort_polypharmacy_with_all_outcome_df.to_csv(f"{output_dir}/data/cohort_polypharmacy_with_outcome.csv", index=False)


In [ ]:
cohort_polypharmacy_with_all_outcome_df= pd.read_csv(f"{output_dir}/data/cohort_polypharmacy_with_outcome.csv")


In [ ]:
def mean_var(df, column_name="concurrent_polypharmacy"):
    ave = df[column_name].mean()
    var = df[column_name].var()
    return (ave, var, var/ave)

In [ ]:
mean_var(cohort_polypharmacy_with_all_outcome_df, 'concurrent_polypharmacy')


In [ ]:
mean_var(cohort_polypharmacy_with_all_outcome_df, 'CCI')


In [ ]:
mean_var(cohort_polypharmacy_with_all_outcome_df, 'ERV_count')


In [ ]:
mean_var(cohort_polypharmacy_with_all_outcome_df, 'IV_count')
